# Forecasting the first positive sepsis label
**PhysioNet Challenge 2019 · Cross-site evaluation · Kaggle GPU**

At the end of each ICU hour, two models estimate the probability of a first positive benchmark label within **6 hours** and **12 hours**. Site A supplies training and threshold selection. Site B supplies the held-out evaluation. Clinical onset has no independent timestamp in this analysis.

### Run on Kaggle

1. Attach [The PhysioNet Challenge 2019 dataset](https://www.kaggle.com/datasets/farjanayesmin/the-physionet-challenge-2019-dataset) and enable a GPU. Cell 4 can also retrieve the dataset through KaggleHub when internet access is available.
2. In Cell 2, keep `RUN_MODE='full'` for the research configuration or select `'pilot'` for a smaller software check. The full run includes five outer folds and repeated model fits and may exceed one Kaggle session.
3. Run from the top. **Cell 4 should print A = 20,336 and B = 20,000 before hashing.** Nested folders with repeated site names are supported.
4. After Cell 40 completes, download **sepsis_research_results.zip** from Output. It contains trained weights, fitted scalers, stack coefficients, figures in three formats, evaluation tables and the files needed to audit them.

| Cells | Work |
|---|---|
| 1–8 | Configure the experiment, resolve the sites, validate the records and test causal features |
| 9–15 | Build the cohort, split by record and verify the temporal network |
| 16–22 | Train the interval models, fit the stack and freeze predictions |
| 23–39 | Evaluate alert policies, calibration, uncertainty, ablations and checkpoint replay |
| 40 | Verify and export the results archive |

[Upstream release and data dictionary](https://physionet.org/content/challenge-2019/1.0.0/)


## Model architecture

| Stage | Input | Operation | Output |
|---|---|---|---|
| Shared history | Hourly PSV rows up through prediction hour *t* | Clean, order by ICU hour and retain measurement flags | 34 dynamic channels plus age and coded sex |
| XGBoost branch | Latest values, flags, measurement ages, two static values and trailing 6/24-hour summaries | Fit one histogram booster per interval | Two interval probabilities |
| Temporal branch | Up to 48 hours × 102 channels, a padding mask and four static inputs | Project 102 channels to 64. Apply four residual blocks with dilations 1, 2, 4 and 8, each with two causal convolutions of width 3. Concatenate the final 64-channel state with the static inputs and apply a 68 → 32 → 2 head | Two interval probabilities |
| Stacking | Tree and temporal logits for the same record, hour and interval | Fit one L2 logistic regression for each interval on out-of-fold predictions from A | First-interval risk *a* and conditional second-interval risk *b* |
| Forecast and alert | Two stacked probabilities | Calculate 6-hour risk as *a* and 12-hour risk as *a + (1 − a)b*. Select thresholds on the reserved 15% of site A and apply a six-hour cooldown | Frozen risks and alert episodes for evaluation on B |

**What the second interval means.** A training row contributes to the second head when the record is known to be transition-free through hour 6 and its 12-hour outcome is known. An event by hour 6 already counts toward the 12-hour forecast.

**What each network state can see.** Left padding and a valid-hour mask prevent the convolution from using later hours. Its theoretical receptive field spans 61 hourly positions, and its input contains at most 48 real positions.

**How patients stay separated.** Each outer fold supplies predictions for records excluded from its training and early stopping. Those predictions train the stack. Cell 38 exports the fitted system’s architecture figure with the other study graphs.


## Cell 1 — Kaggle dependencies and runtime imports

**Purpose.** Import the numerical, tabular, neural, boosting, metrics and visualization libraries. Check the GPU and versions before consuming data. **Why install conditionally?** A default Kaggle environment usually includes PyTorch, XGBoost, NumPy and pandas, but `pyarrow` (Parquet) and `kagglehub` are not guaranteed everywhere; only missing packages are installed. The notebook uses Matplotlib's non-interactive backend for file figures so a PNG/PDF is saved even if no chart is displayed.

**Critical versions:** XGBoost supports `device='cuda',tree_method='hist'` in recent releases; a fallback to CPU is explicitly logged if GPU booster initialization fails. The torch model uses CUDA when available. `tqdm.auto` presents notebook progress bars for record extraction, folds, epochs, policy search and bootstrap. `numba` optionally accelerates deterministic per-patient alert replay. No model results are asserted by this cell.

In [ ]:
import os, sys, json, math, time, gzip, shutil, hashlib, warnings, subprocess, platform, random
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from importlib.util import find_spec

for pkg, module in [('pyarrow', 'pyarrow'), ('numba', 'numba')]:
    if find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.titleweight': 'bold', 'axes.labelcolor': '#23384a',
                     'axes.titlecolor': '#193d54', 'figure.facecolor': 'white'})
import xgboost as xgb
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, roc_auc_score, roc_curve,
                             precision_recall_curve, brier_score_loss, log_loss)
from sklearn.calibration import calibration_curve
from scipy.special import expit, logit
from scipy.stats import iqr
from IPython.display import display, FileLink
try:
    from numba import njit
    HAVE_NUMBA=True
except ImportError:
    HAVE_NUMBA=False
    def njit(*args, **kwargs):
        return lambda fn: fn

print('Python:', sys.version.split()[0], '| Torch:', torch.__version__,
      '| XGBoost:', xgb.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0), '| memory GiB:',
          round(torch.cuda.get_device_properties(0).total_memory / 2**30,2))


## Cell 2 — Freeze the experiment configuration before reading outcome-dependent performance

**Research choice:** `RUN_MODE='full'` runs both public sites and the fivefold outer pipeline. Set `pilot` only to debug a smaller sample from each site. Pilot metrics do not support publication. This notebook deliberately has **no patient-hour subsampling in full mode**, because changing the negative-hour distribution alters AP and event burden.

**Primary values from the protocol:** 48-hour lookback, 6/12-hour event horizons, 5 grouped OOF folds, 15% A-only policy set, depths {3,4,5}, 500 XGB rounds max, patience 40, LR 0.05, TCN batch 128, max epochs 30, patience 5, and 2,000 record bootstrap resamples. All stochastic seeds begin at 2026. `N_WORKERS=0` is intentionally conservative on Kaggle: multi-process copying of large hourly arrays can exhaust memory; increase only after verifying memory.

**Optional comparisons.** Enable `RUN_TREE_ABLATIONS`, `RUN_TCN_ABLATIONS` or `RUN_TWO_TREE_STACK` in the configuration. Reverse-site replication and three-seed stability would require separate experiments. Pilot sampling draws records randomly within each site using the configured seed.

In [ ]:
@dataclass(frozen=True)
class CFG:
    RUN_MODE: str = 'full'               # change to 'pilot' for first debugging run
    SEED: int = 2026
    LOOKBACK: int = 48
    HORIZONS: tuple = (6,12)
    POLICY_FRAC: float = 0.15
    N_FOLDS: int = 5
    INNER_STOP_FRAC: float = 0.15
    PILOT_RECORDS_A: int = 800
    PILOT_RECORDS_B: int = 400
    XGB_DEPTHS: tuple = (3,4,5)
    XGB_ROUNDS: int = 500
    XGB_PATIENCE: int = 40
    XGB_LR: float = 0.05
    XGB_MIN_CHILD: int = 5
    XGB_SUBSAMPLE: float = 0.8
    XGB_COLSAMPLE: float = 0.8
    XGB_L2: float = 1.0
    XGB_THREADS: int = 4
    XGB_USE_GPU: bool = False
    TCN_BATCH: int = 128
    TCN_EPOCHS: int = 30
    TCN_PATIENCE: int = 5
    TCN_LR: float = 0.001
    TCN_WEIGHT_DECAY: float = 0.01
    TCN_DROPOUT: float = 0.1
    TCN_WIDTH: int = 64
    TCN_N_WORKERS: int = 0
    BOOTSTRAPS: int = 2000
    ALERT_FALSE_BUDGET: float = 2.0
    ALERT_COOLDOWN_HOURS: int = 6
    RUN_TREE_ABLATIONS: bool = True
    RUN_TCN_ABLATIONS: bool = True
    RUN_TWO_TREE_STACK: bool = False
    STRICT_INVENTORY: bool = True
    DATASET_HANDLE: str = 'farjanayesmin/the-physionet-challenge-2019-dataset'
    AUTO_ATTACH_DATASET: bool = True       # use KaggleHub if the dataset is not already mounted
    DATA_MOUNT: str = ''                   # optional explicit path; leave empty for automatic resolution
    EXPERIMENT_ID: str = 'xgb_tcn_ablation_cpu_A_to_B_2026'

cfg=CFG()
assert cfg.RUN_MODE in ('pilot','full') and cfg.LOOKBACK == 48
BASE=Path('/kaggle/working') if Path('/kaggle').exists() else Path('/mnt/data')
import tempfile
BASE.mkdir(parents=True, exist_ok=True)
TEMP=Path('/kaggle/temp') if Path('/kaggle').exists() else Path(tempfile.gettempdir())
work_parent=TEMP/'sepsis_research_runtime'
work_parent.mkdir(parents=True, exist_ok=True)
RUN=Path(tempfile.mkdtemp(prefix=cfg.EXPERIMENT_ID+'_', dir=work_parent))
for folder in ('audit','data','fit','scores','figures','tables','policy','models','supplement'):
    (RUN/folder).mkdir(parents=True,exist_ok=True)
OUTPUT_ZIP=BASE/'sepsis_research_results.zip'
with (RUN/'config.json').open('w') as f: json.dump(asdict(cfg),f,indent=2)
SEED=cfg.SEED
np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Mode:',cfg.RUN_MODE,'| device:', DEVICE, '| dedicated temp:', RUN)


## Cell 3 — Immutable raw field schema and output/hashing helpers

**Inputs.** Each PhysioNet PSV holds 40 predictor columns plus `SepsisLabel`; the 34 changing clinical columns are eight vitals and 26 laboratories. `Age` and `Gender` are the two included static values. `Unit1`, `Unit2`, `HospAdmTime`, `ICULOS` and identifiers are metadata/diagnostic fields—not model features. The true time column is allowed to construct windows/targets, but *never* enters a primary predictor matrix.

**Outputs.** This cell makes small reusable functions for SHA-256, deterministic JSON, chunked Parquet and version metadata. `save_fig` writes **three journal-friendly formats** (high-resolution PNG, editable-vector PDF and SVG). File dates and data hashes are for provenance, not model inputs. Stable sorted column names and keys prevent mismatched schemas when checking checkpoints.

In [ ]:
VITALS=['HR','O2Sat','Temp','SBP','MAP','DBP','Resp','EtCO2']
LABS=['BaseExcess','HCO3','FiO2','pH','PaCO2','SaO2','AST','BUN',
      'Alkalinephos','Calcium','Chloride','Creatinine','Bilirubin_direct',
      'Glucose','Lactate','Magnesium','Phosphate','Potassium','Bilirubin_total',
      'TroponinI','Hct','Hgb','PTT','WBC','Fibrinogen','Platelets']
DYN=VITALS+LABS
STATIC=['Age','Gender']
ADMIN=['Unit1','Unit2','HospAdmTime','ICULOS']
RAW_COLS=DYN+STATIC+ADMIN+['SepsisLabel']
assert len(VITALS)==8 and len(LABS)==26 and len(DYN)==34 and len(RAW_COLS)==41
CARRY=np.array([4]*8+[24]*26,dtype=np.int16)

def sha256_file(path,chunk=2**20):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(chunk),b''): h.update(b)
    return h.hexdigest()

def json_dump(obj,path):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    with path.open('w',encoding='utf-8') as f:
        json.dump(obj,f,indent=2,sort_keys=True,default=lambda x: x.item() if isinstance(x,np.generic) else str(x))

def save_fig(name):
    plt.tight_layout()
    for ext in ('png','svg','pdf'):
        plt.savefig(RUN/'figures'/f'{name}.{ext}',dpi=220,bbox_inches='tight')
    plt.close()

ENV={'utc':datetime.now(timezone.utc).isoformat(),'python':sys.version,
     'torch':torch.__version__,'xgboost':xgb.__version__,
     'pandas':pd.__version__,'numpy':np.__version__,
     'hardware':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
     'platform':platform.platform()}
json_dump(ENV,RUN/'environment.json')
print('Expected 512 predictor columns:',34*3+2+34*2*6)


## Cell 4 — Acquire the exact Kaggle dataset and resolve its A/B folders

**Configured source:** [The PhysioNet Challenge 2019 dataset on Kaggle](https://www.kaggle.com/datasets/farjanayesmin/the-physionet-challenge-2019-dataset), handle `farjanayesmin/the-physionet-challenge-2019-dataset`. This is the source requested for this study; the upstream reference is [PhysioNet/CinC Challenge 2019 v1.0.0](https://physionet.org/content/challenge-2019/1.0.0/).

**Acquisition order.** (1) Respect an explicit `CFG.DATA_MOUNT` path. (2) Respect `SEPSIS_DATA_ROOT`. (3) Search existing Kaggle inputs for the exact dataset slug, including owner-nested mount layouts. (4) If not mounted and `AUTO_ATTACH_DATASET=True`, call `kagglehub.dataset_download(CFG.DATASET_HANDLE)`. In a Kaggle Notebook, KaggleHub can automatically attach the public dataset and return its resolved directory. The first-time attachment requires Internet access. If Internet is disabled or KaggleHub cannot attach it, add the same dataset through the right panel **Input → Add Input**, then rerun this cell. Kaggle's notebook documentation also describes adding datasets from the Input panel; KaggleHub's integration can attach resources from code.

**Why this resolver replaces the previous one.** The previous cell searched only immediate children and used a brittle glob pattern. It could miss valid nested layouts such as `training_setA/training_setA/*.psv` and `training_setB/training_setB/*.psv`, and it failed immediately when no input was attached. This resolver searches path ancestry, finds the common dataset root above the first A/B folder, and requires both sites in the same root. It does not silently combine multiple copies or unrelated datasets.

**Expected reconciliation targets:** A=20,336 and B=20,000 records. These are checked in full mode against the protocol; if the mirror differs, inspect the manifest and reconcile the difference rather than silently changing the cohort. The source files remain read-only. Run this cell interactively once before **Save Version**, because Kaggle requires data sources to be attached before a saved-version execution.


In [ ]:
DATASET_SLUG = cfg.DATASET_HANDLE.rsplit('/', 1)[-1]
SITE_DIRS = {'training_seta': 'A', 'training_setb': 'B'}


def _site_root_candidates(scan_root):
    """Return dataset roots containing both A and B PSV files under site folders."""
    scan_root = Path(scan_root).expanduser().resolve()
    if not scan_root.is_dir():
        return {}
    by_root = defaultdict(lambda: {'A': [], 'B': []})
    for p in scan_root.rglob('*.psv'):
        try:
            rel_parts = p.relative_to(scan_root).parts
        except ValueError:
            continue
        site_positions = [(i, SITE_DIRS[part.lower()])
                          for i, part in enumerate(rel_parts[:-1])
                          if part.lower() in SITE_DIRS]
        sites = {site for _, site in site_positions}
        # A record must belong to exactly one source set. Repeated same-site
        # folder names (training_setA/training_setA) are permitted.
        if len(sites) > 1:
            raise ValueError(f"Conflicting site folders: {p}")
        if not sites:
            continue
        site = next(iter(sites))
        first_site_idx = min(i for i, s in site_positions if s == site)
        dataset_root = scan_root.joinpath(*rel_parts[:first_site_idx]).resolve()
        by_root[dataset_root][site].append(p)
    return {root: groups for root, groups in by_root.items()
            if groups['A'] and groups['B']}


def _choose_dataset_root(scan_roots):
    """Resolve exactly one dataset root; prefer the exact configured slug."""
    found = {}
    for scan_root in scan_roots:
        if scan_root and Path(scan_root).is_dir():
            found.update(_site_root_candidates(scan_root))
    if not found:
        return None
    exact = [root for root in found if root.name.lower() == DATASET_SLUG.lower()]
    candidates = exact if len(exact) == 1 else list(found)
    if len(candidates) != 1:
        details = {str(root): {k: len(v) for k, v in groups.items()}
                   for root, groups in found.items()}
        raise RuntimeError(
            'More than one PhysioNet A/B dataset root was found. Remove duplicate '
            f'attachments or set CFG.DATA_MOUNT explicitly. Candidates: {details}'
        )
    root = candidates[0]
    return root, found[root]


def discover_dataset():
    # 1. Explicit user path has highest priority.
    if cfg.DATA_MOUNT:
        root = _choose_dataset_root([Path(cfg.DATA_MOUNT)])
        if root is None:
            raise FileNotFoundError(
                f'CFG.DATA_MOUNT={cfg.DATA_MOUNT!r} does not contain both '
                'training_setA and training_setB PSV files.'
            )
        print('Dataset source: CFG.DATA_MOUNT')
        return root

    # 2. Environment override, then 3. existing Kaggle input mounts.
    env_root = os.environ.get('SEPSIS_DATA_ROOT', '').strip()
    input_root = Path('/kaggle/input')
    scan_roots = []
    if env_root:
        resolved = _choose_dataset_root([Path(env_root)])
        if resolved is None:
            raise FileNotFoundError(f'No paired site folders under SEPSIS_DATA_ROOT={env_root}')
        return resolved
    if input_root.is_dir():
        # Prefer directories whose final component is the exact dataset slug.
        # This supports both /kaggle/input/<slug> and owner-nested layouts.
        exact_mounts = [p for p in input_root.rglob(DATASET_SLUG) if p.is_dir()]
        scan_roots.extend(exact_mounts)
        if not exact_mounts:
            scan_roots.append(input_root)
    root = _choose_dataset_root(scan_roots)
    if root is not None:
        print('Dataset source: existing mounted input ->', root[0])
        return root

    # 4. Automatically attach/download the exact public Kaggle dataset.
    if cfg.AUTO_ATTACH_DATASET:
        try:
            import kagglehub
            print('Dataset not found in existing inputs. Attaching with KaggleHub: '
                  f'{cfg.DATASET_HANDLE}')
            resolved = Path(kagglehub.dataset_download(cfg.DATASET_HANDLE)).expanduser().resolve()
            root = _choose_dataset_root([resolved])
            if root is None:
                # Some KaggleHub versions return a cache/parent directory.
                root = _choose_dataset_root([resolved.parent])
            if root is not None:
                print('Dataset source: KaggleHub ->', root[0])
                return root
            raise RuntimeError(
                f'KaggleHub returned {resolved}, but no paired training_setA/'
                'training_setB PSV folders were found there.'
            )
        except Exception as exc:
            raise RuntimeError(
                '\nUnable to attach or resolve the configured PhysioNet dataset.\n'
                f'Dataset: https://www.kaggle.com/datasets/{cfg.DATASET_HANDLE}\n'
                'Try these steps in Kaggle: (1) turn Internet ON and rerun this cell; '
                '(2) or open the right-side Input panel → Add Input and attach the '
                'dataset above; (3) rerun this cell. If you already mounted it at a '
                'custom path, set CFG.DATA_MOUNT to the directory containing the '
                'training_setA/training_setB folders.\n'
                f'Underlying error: {type(exc).__name__}: {exc}'
            ) from exc

    raise FileNotFoundError(
        'Dataset is not attached. Enable CFG.AUTO_ATTACH_DATASET or attach '
        f'https://www.kaggle.com/datasets/{cfg.DATASET_HANDLE} via Input → Add Input.'
    )


def inventory_entries(groups):
    entries = []
    seen = set()
    for site in ('A', 'B'):
        for path in sorted(groups[site]):
            key = (site, path.stem)
            if key in seen:
                raise ValueError(f'Duplicate site/record key {key} at {path}')
            seen.add(key)
            entries.append((site, path))
    return entries

DATA_ROOT, site_files = discover_dataset()
entries = inventory_entries(site_files)
site_counts = dict(Counter(site for site, path in entries))
print('Pre-hash inventory:', site_counts)
expected = {'A': 20336, 'B': 20000}
if cfg.STRICT_INVENTORY and cfg.RUN_MODE == 'full' and site_counts != expected:
    raise RuntimeError(f'At {DATA_ROOT}, expected {expected} but discovered {site_counts}. Check the attached dataset version and folder layout.')
raw_paths = {(site, path.stem): path for site, path in entries}
from concurrent.futures import ThreadPoolExecutor

def hash_entry(entry):
    site, path = entry
    return {'site': site, 'record': path.stem, 'path': str(path.relative_to(DATA_ROOT)),
            'bytes': path.stat().st_size, 'sha256': sha256_file(path)}

# Concurrent file reads overlap mount latency. map preserves inventory order.
with ThreadPoolExecutor(max_workers=8) as pool:
    rows = list(tqdm(pool.map(hash_entry, entries), total=len(entries), desc='Inventory + SHA-256'))
manifest = pd.DataFrame(rows).sort_values(['site', 'record']).reset_index(drop=True)
manifest.to_parquet(RUN/'data/raw_manifest.parquet', index=False)
inv_hash = hashlib.sha256(
    manifest[['site', 'record', 'sha256']].to_csv(index=False).encode()
).hexdigest()
json_dump({'dataset_handle': cfg.DATASET_HANDLE,
           'dataset_url': f'https://www.kaggle.com/datasets/{cfg.DATASET_HANDLE}',
           'resolved_root': str(DATA_ROOT), 'site_counts': site_counts,
           'inventory_hash': inv_hash,
           'upstream_release': 'PhysioNet/CinC 2019 v1.0.0',
           'mirror_version': 'not available from local mount; record externally if known'},
          RUN/'data/source.json')
print('Source inventory:', site_counts, '| inventory SHA256:', inv_hash)


## Cell 5 — Untrusted PSV parser, chronology validation, and quarantine

**Required parsing contract.** Read the `|` header; require exactly 41 unique names; convert numeric entries with `errors='coerce'` and audit malformed tokens. Replace numeric infinities with missing values. Reject noninteger ICU hours and conflicting duplicate-hour rows. Identical duplicate rows may be de-duplicated and counted. Sort by hour; reindex missing integer hours but keep **missing labels unknown**, never filling them to zero. If a label gap occurs, the later segment is not eligible for the **primary uninterrupted risk set**. Preserve all original clinical values (no undocumented clipping or unit conversion).

**Causal feature source:** The ingestion function stores raw 34-channel dynamic values and two static channels separately; predictor construction never sees label arrays. Missingness flags are based on **valid raw observations** before any carrying. A first-hour-positive record is reported as left-censored/prevalent. Source and filename are only keys. For privacy, generated outputs contain these public record stems only, no real-world patient identity claims.

In [ ]:
def read_record(path):
    with open(path,encoding='utf-8-sig') as f:
        hdr=f.readline().strip().split('|')
    if len(hdr)!=41 or len(set(hdr))!=41 or set(hdr)!=set(RAW_COLS):
        raise ValueError(f'Unexpected PSV header: {path.name}')
    d=pd.read_csv(path,sep='|',dtype=str,low_memory=False)
    malformed=0
    for col in RAW_COLS:
        raw=d[col]
        parsed=pd.to_numeric(raw,errors='coerce')
        malformed+=int((raw.notna() & parsed.isna() & ~raw.str.strip().str.lower().isin(['nan','','na','none'])).sum())
        d[col]=parsed.replace([np.inf,-np.inf],np.nan)
    if d.empty:
        raise ValueError('Empty patient record')
    hour=d['ICULOS'].to_numpy(dtype=np.float64)
    if (not np.isfinite(hour).all()) or (not np.array_equal(hour,np.round(hour))):
        raise ValueError('Nonfinite/fractional ICULOS')
    d=d.sort_values('ICULOS',kind='stable')
    duplicates=int(d['ICULOS'].duplicated().sum())
    if duplicates:
        for _,g in d.groupby('ICULOS',sort=False):
            if len(g)>1:
                x=g[RAW_COLS].to_numpy(dtype=np.float64)
                if not np.all((x==x[0]) | (np.isnan(x)&np.isnan(x[0]))):
                    raise ValueError('Conflicting duplicate ICU hour')
        d=d.drop_duplicates(subset='ICULOS',keep='first')
    hours=d['ICULOS'].astype(int).to_numpy()
    d=d.set_index(d['ICULOS'].astype(int)).reindex(range(int(hours[0]),int(hours[-1])+1))
    values=d[DYN].to_numpy(dtype=np.float32,copy=True)
    static=d[STATIC].to_numpy(dtype=np.float32,copy=True)
    labels=d['SepsisLabel'].to_numpy(dtype=np.float32,copy=True)
    labels[~np.isin(labels,[0.,1.])]=np.nan
    return {'dyn':values,'static':static,'labels':labels,
            'hours':np.arange(int(hours[0]),int(hours[-1])+1,dtype=np.int32),
            'malformed_tokens':malformed,'identical_duplicates':duplicates,
            'inserted_hour_bins':len(d)-len(hours)}


## Cell 6 — Exact risk-set targets, independent from predictors

At a known-negative hour `t`, the next 6 hours means **`(t,t+6]`**; the next 12 means `(t,t+12]`. A verified future positive label makes the corresponding target 1 even if the record stops before the horizon ends. An event-free negative target requires fully observed, trustworthy labels through the horizon. Otherwise the target is unknown (`NaN`) and cannot enter supervised loss or AP. The second-interval conditional target `I2` is defined only if `I1==0`, and then equals the 12-hour cumulative target if observed.

**No immortal-time filter:** short trajectories still get scores; we do not require future complete follow-up as a condition to predict. Eligible input hours stop before the first positive label or the first unknown label. Use these synthetic fixtures to test exact boundaries, event after a horizon, censoring, and first-hour positives before touching real data.

In [ ]:
def rolling_targets(t,last_good,event_hour=None):
    assert t<=last_good and (event_hour is None or t<event_hour)
    def y(h):
        if event_hour is not None and event_hour<=t+h: return 1.0
        return 0.0 if last_good>=t+h else np.nan
    y6,y12=y(6),y(12)
    i2=y12 if np.isfinite(y6) and y6==0 else np.nan
    return y6,y12,y6,i2

assert rolling_targets(10,16,16)[:2] == (1.0,1.0)
assert rolling_targets(10,17,17) == (0.0,1.0,0.0,1.0)
assert rolling_targets(10,22,22) == (0.0,1.0,0.0,1.0)
assert rolling_targets(10,23,23) == (0.0,0.0,0.0,0.0)
assert np.isnan(rolling_targets(10,15,None)[0])
assert rolling_targets(10,16,None)[0]==0 and np.isnan(rolling_targets(10,16,None)[1])
print('All conditional-interval target boundary fixtures passed.')


## Cell 7 — Causal 512-column tabular feature engine

This is the exact **union-feature** tree representation. At every hour, the 34 channels contribute `last`, `measured_now`, `age_hours`; append latest age and coded sex (`34×3+2=104`). A valid last reading carries at most 4 hours for a vital and 24 hours for a lab; stale readings stay missing, even though their capped recency is still recorded. Unknown-in-window age uses sentinel **49**. A 6-bin window contains `[t−5,t]`, and a 24-bin window contains `[t−23,t]`. For each variable/window, calculate **mean, min, max, population SD, count of genuine raw measurements, and least-squares slope per hour** (`34×2×6=408`). A singleton slope is missing; zero observations yield a zero count and five missing summaries.

**Numerical implementation:** float64 for means and slope intermediates, final float32 for XGBoost. Cumulative sums and a causal sliding window calculate summaries; missing observations never become fresh measurements through carry. There is no whole-record normalization, no use of future rows, no explicit ICU clock, no length of completed stay, and no ID. Feature order is deterministic and stored in `schema.json` and `feature_dictionary.csv`.

In [ ]:
STATE_COLS=[f'{v}__{suffix}' for v in DYN for suffix in ('last','measured_now','age_hours')]+['Age','Gender']
TRAJ_COLS=[f'{v}__w{w}__{stat}' for v in DYN for w in (6,24)
           for stat in ('mean','min','max','sd','count','slope')]
FEATURE_COLS=STATE_COLS+TRAJ_COLS
assert len(STATE_COLS)==104 and len(TRAJ_COLS)==408 and len(FEATURE_COLS)==512

def last_age_features(raw):
    # This full-prefix function is for tabular states; any age beyond 47
    # is assigned absent because the deployed TCN/tree use a 48-hour lookback.
    n,v=raw.shape
    ok=np.isfinite(raw)
    idx=np.arange(n,dtype=np.int32)[:,None]
    last_ix=np.maximum.accumulate(np.where(ok,idx,-10**7),axis=0)
    age=idx-last_ix
    age=np.where(age<=47,age,49).astype(np.float32)
    col_age=np.where(age<=47,age,49)
    last=np.take_along_axis(raw,np.clip(last_ix,0,n-1),axis=0)
    keep=(col_age<=CARRY[None,:]) & (col_age!=49)
    last=np.where(keep,last,np.nan).astype(np.float32)
    return last,ok.astype(np.float32),age

def forward_static(raw):
    n,v=raw.shape
    ix=np.arange(n,dtype=np.int32)[:,None]
    prior=np.maximum.accumulate(np.where(np.isfinite(raw),ix,-1),axis=0)
    return np.where(prior>=0,np.take_along_axis(raw,np.clip(prior,0,n-1),axis=0),np.nan).astype(np.float32)

def causal_summaries(raw,w):
    from numpy.lib.stride_tricks import sliding_window_view
    n,k=raw.shape
    padded=np.pad(raw.astype(np.float64),((w-1,0),(0,0)),constant_values=np.nan)
    v=np.moveaxis(sliding_window_view(padded,w,axis=0),-1,1)  # n × w × 34
    valid=np.isfinite(v)
    count=valid.sum(axis=1).astype(np.float64)
    v0=np.where(valid,v,0.)
    s=v0.sum(axis=1)
    mean=np.divide(s,count,out=np.full((n,k),np.nan),where=count>0)
    mini=np.where(valid,v,np.inf).min(axis=1); mini[np.isinf(mini)]=np.nan
    maxi=np.where(valid,v,-np.inf).max(axis=1); maxi[np.isinf(maxi)]=np.nan
    ss=(v0*v0).sum(axis=1)
    var=np.divide(ss,count,out=np.zeros((n,k)),where=count>0)-np.nan_to_num(mean,nan=0)**2
    sd=np.sqrt(np.maximum(var,0));sd[count==0]=np.nan
    tm=np.arange(w,dtype=np.float64)[None,:,None]
    tx=(v0*tm).sum(axis=1)
    t_sum=(valid*tm).sum(axis=1)
    t2_sum=(valid*tm**2).sum(axis=1)
    denom=t2_sum-np.divide(t_sum*t_sum,count,out=np.zeros_like(count),where=count>0)
    numer=tx-np.divide(t_sum*s,count,out=np.zeros_like(count),where=count>0)
    slope=np.divide(numer,denom,out=np.full_like(count,np.nan),where=(count>=2)&(denom>1e-9))
    # Per channel: mean, min, max, SD, observed_count, observed-hour slope.
    return np.stack([mean,mini,maxi,sd,count,slope],axis=2).astype(np.float32)

def make_tabular(dyn,static):
    last,mask,age=last_age_features(dyn)
    sta=forward_static(static)
    st=np.concatenate([np.stack([last,mask,age],axis=2).reshape(len(dyn),102),sta],axis=1)
    # Causal window output has channels ordered variable, window, statistic.
    w6=causal_summaries(dyn,6)
    w24=causal_summaries(dyn,24)
    tr=np.stack([w6,w24],axis=2).reshape(len(dyn),408)
    return np.concatenate([st,tr],axis=1).astype(np.float32)

schema=pd.DataFrame([{'name':c,'view':'state' if i<104 else 'trajectory',
                      'clinical_unit':'upstream documented unit',
                      'fit_scope':'raw, no fitted statistic',
                      'missing_semantics':'NaN except measured bit, count and never-age=49'}
                     for i,c in enumerate(FEATURE_COLS)])
schema.to_csv(RUN/'data/feature_dictionary.csv',index=False)
json_dump({'features':FEATURE_COLS,'dynamic':DYN,'static':STATIC,
           'carry_limits':dict(zip(DYN,CARRY.tolist())),
           'lookback':48,'sampling':{'windows':[6,24]},
           'forbidden':['ICULOS','Unit1','Unit2','HospAdmTime','SepsisLabel','site','record']},
           RUN/'data/schema.json')


## Cell 8 — Mandatory time alignment, missingness and future-leakage unit tests

**Worked physiology fixture:** For systolic BP measured at hours 1,3,5 as 120,112,104, hour 6 has last 104, measured flag 0, recency 1, six-hour mean 112, SD ≈ 6.532 and slope −4 per hour. Verify 4-hour carry works but 5-hour carry expires, labels do not affect the input, and changing every observation **after t** leaves the feature at t unchanged. Test a singleton measurement (slope unknown) and an empty history (count zero). These tests reject misleading passed runs before fitting models.

**Tolerance rationale:** float64 intermediate arithmetic cast to float32 yields small rounding; compare with `1e-4`. A naive test that compares exact floating-point strings can produce false failures. The difference between missing and real zero is preserved by the observed mask and recency.

In [ ]:
def test_features():
    raw=np.full((10,34),np.nan,dtype=np.float32)
    raw[[0,2,4],DYN.index('SBP')]=[120,112,104]
    raw[[0,1,3,5],DYN.index('HR')]=[80,82,88,92]
    sta=np.full((10,2),np.nan,dtype=np.float32)
    x=make_tabular(raw,sta)
    z=dict(zip(FEATURE_COLS,x[5]))
    assert z['SBP__last']==104 and z['SBP__measured_now']==0 and z['SBP__age_hours']==1
    assert z['HR__last']==92 and z['HR__measured_now']==1 and z['HR__age_hours']==0
    assert abs(z['SBP__w6__mean']-112)<1e-4
    assert abs(z['SBP__w6__sd']-np.std([120,112,104]))<1e-4
    assert abs(z['SBP__w6__slope']+4)<1e-4
    assert z['SBP__w6__count']==3
    assert np.isnan(x[5,FEATURE_COLS.index('Platelets__w6__slope')])
    assert z['Platelets__w6__count']==0
    assert x[4,FEATURE_COLS.index('SBP__last')]==104
    assert np.isnan(x[9,FEATURE_COLS.index('SBP__last')])
    after=raw.copy();after[6:,:]=1e5
    xa=make_tabular(after,sta)
    np.testing.assert_allclose(xa[:6],x[:6],rtol=1e-5,atol=1e-5,equal_nan=True)
    one=np.full((3,34),np.nan,dtype=np.float32);one[0,0]=5
    q=make_tabular(one,np.full((3,2),np.nan,dtype=np.float32))
    assert q[0,FEATURE_COLS.index('HR__w6__sd')]==0
    assert np.isnan(q[0,FEATURE_COLS.index('HR__w6__slope')])
    return True
assert test_features()
print('Causality, carry-limit, count, slope and masking fixtures PASS.')


## Cell 9 — Convert all eligible records into keyed hourly and tabular stores

**This is the potentially largest CPU/data stage.** Iterate over the source manifest with a `tqdm` progress bar, read records, audit labels, and construct exactly one tabular row per **at-risk (known-negative) hour**. Keep A/B identities separated by a key. Every record contributes its raw hourly dynamics to a concatenated array for lazy, bounded TCN sequences; it contributes only eligible prefix landmarks to the model matrix. First-positive records are counted but not modeled. Unknown-label interruptions censor follow-up; their later observations cannot reenter primary analysis.

**Memory.** Each record’s 512-column block is appended to a disk-backed float32 matrix. Fitting and prediction still need memory for the selected rows and the model. Approximate storage is 512 × 4 bytes × eligible hours for XGBoost, plus 34 raw channels × 4 bytes × observed hours, label/key metadata and copies required by training. On a 1-million-row example the base feature matrix alone is ≈2.05 GB. In **full** mode no arbitrary row limit or control undersampling changes the task. In **pilot** mode select a deterministic record sample separately by site and keep a separate `pilot` label on every output. Parsing failures enter a QC log; full mode halts on unaccepted schema/chronology failures instead of silently excluding them.

**Censor rule:** a missing label inside the record ends trustworthy follow-up at the preceding hour. A positive label after the first missing-label gap is **not** treated as an observed incident event. Future labels are used *only* in the target creator; feature calculation receives `dyn, static` but never the outcome array.

In [ ]:
if cfg.RUN_MODE=='pilot':
    rr=np.random.default_rng(cfg.SEED)
    selected=[]
    for site,k in [('A',cfg.PILOT_RECORDS_A),('B',cfg.PILOT_RECORDS_B)]:
        sub=manifest.index[manifest.site==site].to_numpy()
        if len(sub)<k: raise RuntimeError(f'Need >= {k} source {site} records for pilot')
        selected.extend(rr.choice(sub,k,replace=False).tolist())
    manifest_run=manifest.loc[sorted(selected)].reset_index(drop=True)
else: manifest_run=manifest

feature_file=RUN/'data/tabular_features.f32'
all_dyn=[]; all_static=[]; all_meta=[]; rec_info=[]; qc=[]
origin=0
with feature_file.open('wb') as feature_stream:
    for entry in tqdm(manifest_run.itertuples(index=False),total=len(manifest_run),desc='Read records / features / labels'):
        key=(entry.site,entry.record); path=raw_paths[key]
        try:
            d=read_record(path)
        except (ValueError, UnicodeError, pd.errors.ParserError) as exc:
            qc.append({'site':entry.site,'record':entry.record,'status':f'ERROR: {exc}'})
            if cfg.RUN_MODE=='full':
                pd.DataFrame(qc).to_csv(RUN/'audit/record_qc_partial.csv',index=False)
                raise
            continue
        n=len(d['hours']); lab=d['labels']
        missing_idx=np.flatnonzero(~np.isfinite(lab))
        stop=int(missing_idx[0])-1 if len(missing_idx) else n-1
        positives=np.flatnonzero(lab[:stop+1]==1) if stop>=0 else np.array([],dtype=int)
        event_idx=int(positives[0]) if len(positives) else None
        prevalent=bool(stop>=0 and lab[0]==1)
        reversal=bool(event_idx is not None and np.any(lab[event_idx+1:stop+1]==0))
        risk_end=min(stop,event_idx-1 if event_idx is not None else stop)
        pos=np.arange(max(0,risk_end+1),dtype=np.int32) if not prevalent else np.array([],dtype=np.int32)
        # We never accept a nonnegative risk hour whose label is unknown or positive.
        if len(pos): assert np.all(lab[pos]==0)
        rid=len(rec_info)
        event_hour=int(d['hours'][event_idx]) if event_idx is not None else -1
        last_good=int(d['hours'][stop]) if stop>=0 else -1
        rec_info.append({'rid':rid,'site':entry.site,'record':entry.record,'event':int(event_idx is not None),
                         'event_hour':event_hour,'last_good':last_good,'length':n,
                         'prevalent':prevalent,'reversal':reversal,'at_risk_rows':len(pos),
                         'origin':origin})
        all_dyn.append(d['dyn']);all_static.append(d['static'])
        if len(pos):
            x=make_tabular(d['dyn'],d['static'])[pos]
            h=d['hours'][pos]
            ys=np.array([rolling_targets(int(t),last_good,event_hour if event_hour>=0 else None)
                         for t in h],dtype=np.float32)
            # The global hour_index points to prefix-only clinical data, not outcomes.
            m=pd.DataFrame({'rid':rid,'site':entry.site,'record':entry.record,
                            'hour':h.astype(np.int32),'hour_index':origin+pos,
                            'event_hour':event_hour,'last_good':last_good,
                            'y6':ys[:,0],'y12':ys[:,1],'i1':ys[:,2],'i2':ys[:,3]})
            x.tofile(feature_stream);all_meta.append(m)
        qc.append({'site':entry.site,'record':entry.record,'status':'ok',
                   'hours':n,'malformed_tokens':d['malformed_tokens'],
                   'duplicate_hours_dropped':d['identical_duplicates'],
                   'inserted_missing_hours':d['inserted_hour_bins'],
                   'label_gaps':int(np.isnan(lab).sum()),'prevalent':prevalent,
                   'reversal':reversal,'event':event_idx is not None,'eligible':len(pos)})
        origin+=n
    
if not all_meta: raise RuntimeError('No eligible landmark rows were created')
meta=pd.concat(all_meta,ignore_index=True)
X=np.memmap(feature_file,dtype=np.float32,mode="r",shape=(len(meta),len(FEATURE_COLS)))
HOURLY=np.concatenate(all_dyn,axis=0); HOURLY_STATIC=np.concatenate(all_static,axis=0)
rec=pd.DataFrame(rec_info); quality=pd.DataFrame(qc)
assert len(X)==len(meta) and X.shape[1]==512
assert not meta.duplicated(['site','record','hour']).any()
assert all(not col in FEATURE_COLS for col in ['ICULOS','SepsisLabel','site','record','HospAdmTime'])
assert np.all(meta['hour_index'].to_numpy()<len(HOURLY))
# No sample has 12h-positive=0 when its 6h target is 1.
seen=meta[['y6','y12']].dropna()
assert np.all(seen.y12.to_numpy()>=seen.y6.to_numpy())
quality.to_csv(RUN/'audit/record_qc.csv',index=False)
rec.to_parquet(RUN/'data/record_registry.parquet',index=False)
meta.to_parquet(RUN/'data/landmarks.parquet',index=False)
np.save(RUN/'data/hourly_clinical.npy',HOURLY)
np.save(RUN/'data/hourly_static.npy',HOURLY_STATIC)
del all_dyn, all_static, all_meta
summary=meta.groupby('site').agg(landmarks=('hour','size'),
    known_y6=('y6','count'),known_y12=('y12','count'),
    y6_positive=('y6','sum'),y12_positive=('y12','sum')).reset_index()
summary.to_csv(RUN/'tables/cohort_landmarks.csv',index=False)
print('Eligible tabular matrix:',X.shape,'; hourly store:',HOURLY.shape)
display(summary)


## Cell 10 — Cohort flow, missingness visualization and research denominators

**The figures reveal which study population is actually being evaluated.** Plot the source-A/B event records, at-risk records and evaluable 6/12-hour landmarks. Plot raw measurement availability by clinical variable and site, not imputed-value availability: a sparse lab does not become a fresh measurement after last-value carry. Persist raw count tables and three image formats. This step is descriptive and cannot be used to select a model based on the held-out site's label/feature association.

**Journal reporting:** give the number of records, incident events, prevalent/left-censored records, interrupted follow-up, valid negative predictions, observed positive landmarks and censored landmarks. Site B is held out for **fitting**, but cohort counts are required to interpret sampling and missingness. The source protocol's old 2,932 ever-positive count is *not* expected to equal this revised incident count.

In [ ]:
cohort=rec.groupby('site').agg(records=('rid','size'),incident_events=('event',lambda s: int((s.astype(bool) & ~rec.loc[s.index,'prevalent']).sum())),
    prevalent=('prevalent','sum'),reversal=('reversal','sum'),
    risk_records=('at_risk_rows',lambda s: int((s>0).sum())),
    eligible_hours=('at_risk_rows','sum')).reset_index()
cohort.to_csv(RUN/'tables/cohort_records.csv',index=False)
missing=[]
for site in ('A','B'):
    subset=rec[rec.site==site]
    if subset.empty: continue
    indices=np.concatenate([np.arange(int(r.origin),int(r.origin+r.length)) for r in subset.itertuples(index=False)])
    valid=np.isfinite(HOURLY[indices]).mean(axis=0)
    missing.extend({'site':site,'variable':d,'observed_fraction':float(v)} for d,v in zip(DYN,valid))
missingness=pd.DataFrame(missing)
missingness.to_csv(RUN/'tables/missingness_by_site.csv',index=False)
display(cohort)
fig,ax=plt.subplots(figsize=(10,5))
mv=missingness.pivot(index='variable',columns='site',values='observed_fraction').reindex(DYN)
ax.imshow(mv.to_numpy(),aspect='auto',vmin=0,vmax=1,cmap='viridis')
ax.set_xticks(np.arange(len(mv.columns)),mv.columns)
ax.set_yticks(np.arange(len(DYN)),DYN,fontsize=7)
ax.set_title('Fraction of raw valid measurements by source (before imputation)')
plt.colorbar(ax.images[0],ax=ax,label='Observed fraction')
save_fig('01_raw_missingness_heatmap')
fig,axes=plt.subplots(1,2,figsize=(11,4))
cohort.set_index('site')[['records','incident_events','prevalent']].plot.bar(ax=axes[0])
axes[0].set_title('Record-level source cohorts'); axes[0].set_ylabel('Records')
summary.set_index('site')[['known_y6','known_y12']].plot.bar(ax=axes[1])
axes[1].set_title('Censor-aware, evaluable landmark denominators'); axes[1].set_ylabel('Hours')
save_fig('02_cohort_flow_counts')


## Cell 11 — Locked record-grouped policy/fitting partition and OOF folds

**Three compartments:** source A is split **by entire record**, 85% fitting pool and 15% policy validation. The fitting records are divided into **five patient-record-disjoint outer folds** with event stratification. All hours and both horizons for a record stay in one compartment. Site B is assigned `heldout` and is not touched by any fit. If a stronger dataset exposes stable person IDs, this protocol requires replacing record grouping with person grouping; no person ID is exposed by the public PSV files.

**Why event stratification?** The incident outcome is uncommon. It is safe to inspect record-level event status to balance **development splits**, but unsafe to use future event status as an input feature. Test every fold for both classes separately for I1 and I2. A small pilot with inadequate event support may halt; increase the pilot records rather than duplicating positive examples. Save split hashes and explicit record assignments for audit.

In [ ]:
rec['split']='heldout';rec['fold']=-1
source_a=rec[(rec.site=='A')&(rec.at_risk_rows>0)].copy()
source_b=rec[(rec.site=='B')&(rec.at_risk_rows>0)].copy()
if len(source_a)<cfg.N_FOLDS*4 or source_b.empty:
    raise RuntimeError('Not enough at-risk A/B records; check mirror and pilot size')
fit_ids,policy_ids=train_test_split(source_a.rid.to_numpy(),test_size=cfg.POLICY_FRAC,
    stratify=source_a.event.to_numpy(),random_state=cfg.SEED)
rec.loc[rec.rid.isin(fit_ids),'split']='fit'
rec.loc[rec.rid.isin(policy_ids),'split']='policy'
fold_table=rec.set_index('rid').loc[fit_ids]
skf=StratifiedKFold(n_splits=cfg.N_FOLDS,shuffle=True,random_state=cfg.SEED)
for fold,(_,j) in enumerate(skf.split(fit_ids,fold_table['event'].to_numpy())):
    rec.loc[rec.rid.isin(fit_ids[j]),'fold']=fold
rec['fold']=rec['fold'].astype(int)
meta['split']=rec.set_index('rid').loc[meta.rid,'split'].to_numpy()
meta['fold']=rec.set_index('rid').loc[meta.rid,'fold'].to_numpy()
assert not set(fit_ids)&set(policy_ids)
assert not set(fit_ids)&set(source_b.rid)
assert not set(policy_ids)&set(source_b.rid)
assert (meta.loc[meta.split=='fit','fold']>=0).all()
for fold in range(cfg.N_FOLDS):
    tr=meta[(meta.split=='fit')&(meta.fold!=fold)]
    va=meta[(meta.split=='fit')&(meta.fold==fold)]
    assert not set(tr.rid)&set(va.rid)
    for tar in ('i1','i2'):
        a=tr[tar].dropna().astype(int)
        if a.nunique()!=2:
            raise RuntimeError(f'Fold {fold}: no binary class support for {tar}; enlarge pilot')
rec.to_parquet(RUN/'data/record_splits.parquet',index=False)
meta.to_parquet(RUN/'data/landmarks.parquet',index=False)
split_hash=hashlib.sha256(rec[['site','record','split','fold']].to_csv(index=False).encode()).hexdigest()
json_dump({'split_hash':split_hash,'site_A_policy_records':len(policy_ids),
           'site_A_fit_records':len(fit_ids),'outer_folds':cfg.N_FOLDS},RUN/'data/split_hash.json')
print(rec.groupby(['site','split','fold']).agg(n_records=('rid','size'),events=('event','sum')))


## Cell 12 — Batch generator for the bounded causal 48-hour sequence

The TCN receives a **48 × 102** hourly array. Each valid timestep has 34 *most recent non-expired values*, 34 *measured-this-hour* flags, and 34 *recencies divided by 49*. Values are standardized using a median/IQR derived from unique raw **training record-hours** only; absent/expired values become standardized zero **after** their mask/age signals are retained. The 48 rows are right-aligned with a 48-bit valid-hour mask; short histories do not get fabricated clinical measurements.

**Subtle leakage guard:** recency resets at the **left edge of each 48-hour sequence**. A reading before `t−47` cannot influence any value or recency at time t. This is why a window is built from the bounded raw prefix rather than cutting a precomputed full-life forward-filled sequence. Samples are assembled on demand from compact hourly matrices, not a `N × 48 × 102` float32 tensor (~20 GB per million samples). This saves Kaggle disk/RAM at the expense of CPU time.

**TCN static:** latest known Age, coded Gender, plus two static-missingness indicators; age is scaled using training-derived median/IQR. `lookback=6` and `lookback=24` can be passed to the same function for **retrained** history ablations.

In [ ]:
class SequenceScaler:
    def fit(self,record_ids):
        # Train-only, UNIQUE AT-RISK record-hours; exclude post-label/follow-up rows.
        hours=meta.loc[meta.rid.isin(record_ids),'hour_index'].to_numpy(dtype=np.int64)
        assert len(np.unique(hours)) == len(hours)
        d=HOURLY[hours]
        with warnings.catch_warnings():
            warnings.simplefilter('ignore',RuntimeWarning)
            self.med=np.nanmedian(d,axis=0).astype(np.float32)
            self.q1=np.nanpercentile(d,25,axis=0).astype(np.float32)
            self.q3=np.nanpercentile(d,75,axis=0).astype(np.float32)
            age=HOURLY_STATIC[hours,0]
            self.age_med=float(np.nanmedian(age));self.age_iqr=float(np.nanpercentile(age,75)-np.nanpercentile(age,25))
        self.med=np.nan_to_num(self.med,nan=0.)
        self.scale=self.q3-self.q1
        self.scale=np.where(np.isfinite(self.scale)&(self.scale>1e-8),self.scale,1.).astype(np.float32)
        self.age_med=0. if not np.isfinite(self.age_med) else self.age_med
        self.age_iqr=1. if not np.isfinite(self.age_iqr) or self.age_iqr<1e-8 else self.age_iqr
        self.fit_rids_hash=hashlib.sha256(np.sort(np.array(list(record_ids),dtype=np.int32)).tobytes()).hexdigest()
        return self
    def dump(self,path):
        json_dump({'med':self.med.tolist(),'scale':self.scale.tolist(),
                   'age_med':self.age_med,'age_iqr':self.age_iqr,
                   'train_record_hash':self.fit_rids_hash},path)

def sequence_at_landmark(hour_index,record_start,static2,scaler,lookback=48,without_observation=False):
    start=max(int(record_start),int(hour_index)-lookback+1)
    r=HOURLY[start:int(hour_index)+1]
    n=len(r);ok=np.isfinite(r)
    ix=np.arange(n,dtype=np.int32)[:,None]
    last_ix=np.maximum.accumulate(np.where(ok,ix,-10**7),axis=0)
    age=np.where(last_ix<0,49,ix-last_ix).astype(np.float32)
    last=np.take_along_axis(r,np.clip(last_ix,0,n-1),axis=0)
    valid=(age<=CARRY[None,:])&(age!=49)
    carried=np.where(valid,last,np.nan)
    x=np.nan_to_num((carried-scaler.med[None,:])/scaler.scale[None,:],nan=0.,posinf=0.,neginf=0.)
    # channel order per variable: value, this-hour mask, age/49
    if without_observation:
        x=np.stack([x,np.zeros_like(x),np.zeros_like(x)],axis=2).reshape(n,102)
    else:
        x=np.stack([x,ok.astype(np.float32),age/49.],axis=2).reshape(n,102)
    result=np.zeros((48,102),dtype=np.float32)
    hour_mask=np.zeros(48,dtype=np.float32)
    result[-n:]=x;hour_mask[-n:]=1
    a,g=static2
    sm=np.array([float(np.isfinite(a)),float(np.isfinite(g))],dtype=np.float32)
    static4=np.array([((a-scaler.age_med)/scaler.age_iqr) if sm[0] else 0.,
                      g if sm[1] else 0.,1-sm[0],1-sm[1]],dtype=np.float32)
    return result,hour_mask,static4

class LandmarkSequenceDataset(Dataset):
    def __init__(self,indices,scaler,lookback=48,without_observation=False):
        self.indices=np.asarray(indices,dtype=np.int64)
        self.scaler=scaler;self.lookback=lookback;self.no_obs=without_observation
        self.hour_indices=meta.hour_index.to_numpy(dtype=np.int32)
        self.record_starts=rec.set_index('rid')['origin'].to_numpy(dtype=np.int32)
        self.rids=meta.rid.to_numpy(dtype=np.int32)
        self.statics=X[:,102:104]
        self.labels=meta[['i1','i2']].to_numpy(dtype=np.float32)
    def __len__(self): return len(self.indices)
    def __getitem__(self,j):
        idx=int(self.indices[j]);rid=int(self.rids[idx])
        seq,mask,static=sequence_at_landmark(self.hour_indices[idx],self.record_starts[rid],
                              self.statics[idx],self.scaler,self.lookback,self.no_obs)
        y=self.labels[idx];known=np.isfinite(y).astype(np.float32)
        return (torch.from_numpy(seq),torch.from_numpy(mask),torch.from_numpy(static),
                torch.from_numpy(np.nan_to_num(y,nan=0)),torch.from_numpy(known))

def loader(indices,scaler,training=False,lookback=48,no_obs=False):
    ds=LandmarkSequenceDataset(indices,scaler,lookback,no_obs)
    return DataLoader(ds,batch_size=cfg.TCN_BATCH,shuffle=training,
        num_workers=cfg.TCN_N_WORKERS,pin_memory=(DEVICE.type=='cuda'),
        persistent_workers=(cfg.TCN_N_WORKERS>0),drop_last=False)


## Cell 13 — TCN architecture, causal convolution proof and parameter count

**Exact frozen network.** A 1×1 projection maps the 102-feature hourly input to 64 channels. Four residual blocks use dilation factors `[1,2,4,8]`; every block contains two **left-only padded** convolutions with kernel size 3, per-hour channel LayerNorm, GELU, dropout 0.1 and an identity residual skip. The sequence valid-hour mask is applied following input projection and each convolution activation and residual block. This prevents padded initial hours from contributing model state. The final 64 channels of the most recent *real* hour are concatenated with age, sex and two static-missingness indicators (total 68); dense layers are `68→32→2`.

**Training objective:** `BCEWithLogits` with the sum of known I1 and I2 losses divided by the number of actually known interval labels in that batch, so censored second-interval labels do not become negatives. No positive-class resampling/weighting in the primary run. A kernel-1 experimental variant cannot learn across-hour context under last-state pooling; it serves as an explicitly retrained order-sensitivity check, not an independently invented model. The dropout mask switches off under `.eval()`.

In [ ]:
class ChannelLayerNorm(nn.Module):
    def __init__(self,c): super().__init__();self.norm=nn.LayerNorm(c)
    def forward(self,x): return self.norm(x.transpose(1,2)).transpose(1,2)

class CausalConv(nn.Module):
    def __init__(self,channels,dilation=1,kernel=3,dropout=.1):
        super().__init__()
        self.pad=(kernel-1)*dilation
        self.conv=nn.Conv1d(channels,channels,kernel_size=kernel,dilation=dilation)
        self.norm=ChannelLayerNorm(channels)
        self.act=nn.GELU();self.drop=nn.Dropout(dropout)
    def forward(self,x,mask):
        x=self.conv(nn.functional.pad(x,(self.pad,0)))
        x=self.drop(self.act(self.norm(x)))
        return x*mask

class ResidualTCNBlock(nn.Module):
    def __init__(self,c,d,kernel=3,dropout=.1):
        super().__init__()
        self.c1=CausalConv(c,d,kernel,dropout)
        self.c2=CausalConv(c,d,kernel,dropout)
    def forward(self,x,mask):
        return (x+self.c2(self.c1(x,mask),mask))*mask

class CausalTCN(nn.Module):
    def __init__(self,kernel=3,dropout=None,width=None):
        super().__init__()
        width=width or cfg.TCN_WIDTH
        dropout=cfg.TCN_DROPOUT if dropout is None else dropout
        self.proj=nn.Conv1d(102,width,1)
        self.blocks=nn.ModuleList([ResidualTCNBlock(width,d,kernel,dropout) for d in (1,2,4,8)])
        self.head=nn.Sequential(nn.Linear(width+4,32),nn.GELU(),nn.Dropout(dropout),nn.Linear(32,2))
    def forward(self,seq,valid_hours,static4,return_states=False):
        mask=valid_hours[:,None,:]
        z=self.proj(seq.transpose(1,2))*mask
        states=[z] if return_states else None
        for block in self.blocks:
            z=block(z,mask)
            if return_states: states.append(z)
        # Valid hours are right-aligned, so the last position is always the index t.
        last=z[:,:,-1]
        logits=self.head(torch.cat([last,static4],dim=1))
        return (logits,states) if return_states else logits

def count_params(model): return sum(p.numel() for p in model.parameters() if p.requires_grad)
net_fixture=CausalTCN().eval()
print('TCN trainable parameters:',count_params(net_fixture),'| receptive field:',1+2*(3-1)*sum((1,2,4,8)))
with torch.no_grad():
    dummy=torch.zeros(2,48,102)
    valid=torch.ones(2,48);valid[0,:35]=0
    pred=net_fixture(dummy,valid,torch.zeros(2,4))
    assert pred.shape==(2,2)
del net_fixture


## Cell 14 — Neural causality and bounded-history tests, independent of outcome data

**Why this is necessary:** a tensor model can look causal while accidentally introducing right-padding, future measurements, or corrupt gradients from absent hours. Check intermediate TCN state at hour `t` after radically modifying all **future** slots: the activation at `t` must be unchanged at every block. Check that changing the contents of padded hours cannot affect the final prediction, and that a short case gives the same output alone or batched beside a long case.

This is a randomly initialized synthetic test, *not* evidence about clinical predictive performance. Passing it is a precondition for the first GPU training run.

In [ ]:
test_net=CausalTCN().eval()
with torch.no_grad():
    gen=torch.Generator().manual_seed(cfg.SEED)
    s=torch.randn(1,48,102,generator=gen)
    st=torch.randn(1,4,generator=gen)
    valid=torch.ones(1,48)
    _,before=test_net(s,valid,st,return_states=True)
    modified=s.clone();modified[:,29:,:]=torch.randn(modified[:,29:,:].shape,generator=gen)*100
    _,after=test_net(modified,valid,st,return_states=True)
    for a,b in zip(before,after):
        assert torch.allclose(a[:,:,:29],b[:,:,:29],atol=1e-5), 'Noncausal convolution detected'
    short_valid=valid.clone();short_valid[:,:31]=0
    padded=s.clone();padded[:,:31,:]=1e5
    assert torch.allclose(test_net(s,short_valid,st),test_net(padded,short_valid,st),atol=1e-5)
    one=test_net(s,valid,st)
    batched=test_net(torch.cat([s,s],0),valid.repeat(2,1),st.repeat(2,1))[:1]
    assert torch.allclose(one,batched,atol=1e-5)
del test_net
print('TCN temporal-causality, padding invariance and batch invariance checks PASS.')


## Cell 15 — Shared logit/probability composition and exact masked losses

The input to each second-stage logistic head is a pair of **base-model logits** (not final cumulative probabilities). Given its fitted first-interval output `a` and second conditional-interval output `b`, map them to `P6=a` and `P12=a+(1−a)b`. The same operation is used for the main stack, the raw XGBoost baseline, the calibrated XGBoost baseline, the standalone TCN and the fixed 50:50 branch average. This keeps comparison keys and target definitions identical.

**Probability clipping:** only apply `[1e−6,1−1e−6]` before `logit` transforms or logarithmic loss; do **not** clip or rescale scores for AP/ranking. Verify exact ordering on a constructed fixture. The masked loss has denominator equal to the number of known interval labels, not batch size; this avoids treating an unknown I2 as a zero and avoids implicitly doubling the importance of records with two labels.

In [ ]:
def clipped_logit(p): return logit(np.clip(np.asarray(p,dtype=np.float64),1e-6,1-1e-6))
def compose_intervals(interval):
    v=np.asarray(interval,dtype=np.float64)
    p6=v[:,0];p12=p6+(1-p6)*v[:,1]
    out=np.column_stack([p6,p12])
    if np.any(out < -1e-8) or np.any(out>1+1e-8) or np.any(out[:,1]+1e-8<out[:,0]):
        raise AssertionError('Impossible cumulative risks')
    return np.clip(out,0,1)

def masked_bce(logits,targets,known):
    z=nn.functional.binary_cross_entropy_with_logits(logits,targets,reduction='none')
    return (z*known).sum()/known.sum().clamp_min(1)

assert np.allclose(compose_intervals(np.array([[.1,.2]])),[[.1,.28]])
fixture_logits=torch.tensor([[0.,0.],[2.,-1.]],dtype=torch.float32)
fixture_targets=torch.tensor([[1.,0.],[0.,0.]])
fixture_known=torch.tensor([[1.,0.],[1.,1.]])
assert torch.isfinite(masked_bce(fixture_logits,fixture_targets,fixture_known)).item()
print('Interval composition P12>=P6 and masked BCE fixture PASS.')


## Cell 16 — Fold-local XGBoost model selection: depth, early stopping, GPU/CPU diagnostics

**Training:** one XGBoost binary booster per conditional interval. Use the **same full 512-feature matrix** for both; missing features remain `NaN` and are routed natively. The prespecified candidate depths `{3,4,5}` share histogram boosting, learning rate 0.05, min-child-weight 5, subsample/feature fraction 0.8, L2=1, no L1/gamma, 500 maximum rounds, log-loss metric and 40-round patience. Classes are left in their natural frequencies (`scale_pos_weight=1`). We explicitly preserve the best `iteration+1` rounds when refitting; the stopping subset is disjoint by record.

**GPU:** request `device='cuda'` with `tree_method='hist'`. Some GPU runtime images include CPU-only XGBoost; in that case a specific error is captured and a CPU retry plus hardware warning is written in the manifest. This does **not** change train/test partition or selected model family. `XGBoost` progress bars advance by boosting iteration. Internal candidate records, stopping rounds, best loss, parameters, device and elapsed fitting time are serialized for reproducibility.

In [ ]:
TRAIN_A=np.flatnonzero(meta.split.to_numpy()=='fit')
POLICY_A=np.flatnonzero(meta.split.to_numpy()=='policy')
SITE_B=np.flatnonzero(meta.site.to_numpy()=='B')
ALL_LABELS=meta[['i1','i2']].to_numpy(dtype=np.float32)
META_RID=meta.rid.to_numpy(dtype=np.int32)
FOLD=meta.fold.to_numpy(dtype=np.int8)

class XGBProgress(xgb.callback.TrainingCallback):
    def __init__(self,desc,total):
        self.bar=tqdm(total=total,desc=desc,leave=False)
        self.previous=0
    def after_iteration(self,model,epoch,evals_log):
        current=epoch+1
        if current-self.previous>=5:
            self.bar.update(current-self.previous);self.previous=current
        return False
    def close(self): self.bar.close()

xgb_device='cuda' if cfg.XGB_USE_GPU and torch.cuda.is_available() else 'cpu'
XGB_DEVICE_LOG=[]
def xgb_params(depth,seed,device=None):
    return {'objective':'binary:logistic','eval_metric':'logloss','tree_method':'hist',
            'device':device or xgb_device,'max_depth':int(depth),'eta':cfg.XGB_LR,
            'min_child_weight':cfg.XGB_MIN_CHILD,'subsample':cfg.XGB_SUBSAMPLE,
            'colsample_bytree':cfg.XGB_COLSAMPLE,'reg_lambda':cfg.XGB_L2,
            'reg_alpha':0,'gamma':0,'scale_pos_weight':1,'max_bin':256,
            'seed':int(seed),'nthread':cfg.XGB_THREADS}

def fit_booster(idx,ycol,depth,rounds,seed,stop_idx=None,description='XGB'):
    idx=np.asarray(idx,dtype=np.int64); y=ALL_LABELS[idx,ycol]
    known=np.isfinite(y);idx=idx[known];y=y[known]
    if len(np.unique(y))!=2: raise RuntimeError(f'{description}: only one class in fit interval {ycol}')
    dtrain=xgb.DMatrix(X[idx],label=y,feature_names=FEATURE_COLS,missing=np.nan)
    evals=[]
    if stop_idx is not None:
        sy=ALL_LABELS[stop_idx,ycol];good=np.isfinite(sy);stop_idx=stop_idx[good]
        sy=sy[good]
        if len(np.unique(sy))!=2: raise RuntimeError(f'{description}: only one class in stop set, interval {ycol}')
        evals=[(xgb.DMatrix(X[stop_idx],label=sy,feature_names=FEATURE_COLS,missing=np.nan),'stop')]
    global xgb_device
    progress=XGBProgress(description,rounds)
    params=xgb_params(depth,seed)
    started=time.monotonic()
    try:
        booster=xgb.train(params,dtrain,num_boost_round=int(rounds),evals=evals,
            early_stopping_rounds=cfg.XGB_PATIENCE if evals else None,
            callbacks=[progress],verbose_eval=False)
        used=params['device']
    except xgb.core.XGBoostError as exc:
        progress.close()
        if params['device']!='cuda': raise
        warnings.warn(f'CUDA XGBoost failed; explicit fallback to CPU: {str(exc)[:200]}')
        XGB_DEVICE_LOG.append(str(exc));xgb_device='cpu'
        params=xgb_params(depth,seed,device='cpu')
        progress=XGBProgress(description+' CPU retry',rounds)
        booster=xgb.train(params,dtrain,num_boost_round=int(rounds),evals=evals,
            early_stopping_rounds=cfg.XGB_PATIENCE if evals else None,
            callbacks=[progress],verbose_eval=False)
        used='cpu'
    finally: progress.close()
    iters=int(booster.best_iteration+1) if evals else int(rounds)
    best=float(booster.best_score) if evals else float('nan')
    return booster,{'depth':int(depth),'rounds':iters,'stop_logloss':best,
                    'interval':int(ycol+1),'device':used,
                    'duration_sec':time.monotonic()-started,
                    'n_fit_landmarks':int(len(idx)),
                    'n_stop_landmarks':int(len(stop_idx)) if stop_idx is not None else 0}

def predict_xgb(booster,idx,batch=250000):
    out=np.empty(len(idx),dtype=np.float32)
    for start in range(0,len(idx),batch):
        s=idx[start:start+batch]
        d=xgb.DMatrix(X[s],feature_names=FEATURE_COLS,missing=np.nan)
        out[start:start+batch]=booster.predict(d)
    return out

def grouped_inner_split(rids,seed):
    ids=np.array(sorted(set(map(int,rids))),dtype=np.int32)
    ev=rec.set_index('rid').loc[ids,'event'].to_numpy()
    left,right=train_test_split(ids,test_size=cfg.INNER_STOP_FRAC,stratify=ev,random_state=seed)
    return left,right

def ix_by_rids(record_ids):
    return np.flatnonzero(np.isin(META_RID,np.asarray(record_ids,dtype=np.int32)))

def select_xgb(train_idx,stop_idx,seed,tag,subset=None):
    # Optionally overwrite feature subset in a separately named ablation function.
    chosen=[];all_candidates=[]
    for target_col in (0,1):
        candidates=[]
        for depth in cfg.XGB_DEPTHS:
            model,info=fit_booster(train_idx,target_col,depth,cfg.XGB_ROUNDS,seed,stop_idx,
                                   f'{tag}: interval {target_col+1} depth={depth}')
            candidates.append(info);all_candidates.append(dict(info,tag=tag))
            del model
        infos=candidates
        good=sorted(infos,key=lambda d:(d['stop_logloss'],d['depth'],d['rounds']))[0]
        chosen.append(good)
    return chosen,all_candidates


## Cell 17 — Complete TCN trainer: AdamW, early stopping, refit, gradients and progress

For each outer fold, a **grouped inner validation** set chooses the best training epoch (at most 30; patience 5 on masked interval log loss), and the chosen epoch count is then used to **refit from scratch on all non-held-fold records** before OOF scoring. This is essential: validation records that controlled early stopping cannot also count as unseen OOF records for the stack. Use AdamW, learning rate `0.001`, weight decay `0.01`, batch `128`, gradient clipping `1.0` and a two-output masked loss. Batch/epoch bars track learning; JSON logs preserve each epoch's training and validation losses, selected epoch and training time.

The scaler is fitted on distinct training hours only. A separate fresh scaler is fitted on the entire refit partition. The serializable PyTorch checkpoint contains `state_dict`, architecture, seed and record ID hash; the matching median/IQR JSON is saved next to it. Predictions use `.eval()` and `torch.no_grad()`. If CUDA runs out of memory, reduce `TCN_BATCH` and start a new run with the changed config; do **not** mix half-trained weights into the report.

In [ ]:
TCN_HISTORY=[]
def tcn_evaluate_loss(net,dl):
    net.eval();numer=0.;denom=0.
    with torch.no_grad():
        for seq,hmask,sta,y,known in dl:
            seq=seq.to(DEVICE,non_blocking=True);hmask=hmask.to(DEVICE,non_blocking=True)
            sta=sta.to(DEVICE,non_blocking=True);y=y.to(DEVICE);known=known.to(DEVICE)
            logits=net(seq,hmask,sta)
            loss=nn.functional.binary_cross_entropy_with_logits(logits,y,reduction='none')
            numer+=float((loss*known).sum().item());denom+=float(known.sum().item())
    return numer/denom if denom else np.nan

def tcn_train(indices,scaler,seed,tag,validation_indices=None,fixed_epochs=None,
              kernel=3,lookback=48,no_obs=False):
    torch.manual_seed(seed);np.random.seed(seed);random.seed(seed)
    net=CausalTCN(kernel=kernel).to(DEVICE)
    opt=torch.optim.AdamW(net.parameters(),lr=cfg.TCN_LR,weight_decay=cfg.TCN_WEIGHT_DECAY)
    train_dl=loader(indices,scaler,training=True,lookback=lookback,no_obs=no_obs)
    valid_dl=loader(validation_indices,scaler,training=False,lookback=lookback,no_obs=no_obs) if validation_indices is not None else None
    cap=fixed_epochs if fixed_epochs is not None else cfg.TCN_EPOCHS
    early=0;best=float('inf');best_epoch=0
    log=[];t0=time.monotonic()
    ckpt=RUN/'fit'/f'{tag}.pt'
    for epoch in tqdm(range(1,int(cap)+1),desc=tag+' TCN epochs'):
        net.train();total=0.;nlabels=0
        for seq,hmask,sta,y,known in train_dl:
            if known.sum().item()==0: continue
            seq=seq.to(DEVICE,non_blocking=True);hmask=hmask.to(DEVICE,non_blocking=True)
            sta=sta.to(DEVICE,non_blocking=True);y=y.to(DEVICE);known=known.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logits=net(seq,hmask,sta)
            loss=masked_bce(logits,y,known)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(),max_norm=1.0)
            opt.step()
            w=float(known.sum().item());total+=float(loss.item())*w;nlabels+=w
        train_loss=total/max(nlabels,1.)
        val_loss=tcn_evaluate_loss(net,valid_dl) if valid_dl is not None else np.nan
        log.append({'tag':tag,'epoch':epoch,'train_masked_logloss':train_loss,
                    'valid_masked_logloss':val_loss,'known_train_interval_labels':int(nlabels)})
        if valid_dl is None or val_loss<best-1e-10:
            best=val_loss if valid_dl is not None else train_loss
            best_epoch=epoch;early=0
            torch.save({'state_dict':{key:val.detach().cpu().clone() for key,val in net.state_dict().items()},'kernel':kernel,
                        'lookback':lookback,'without_observation':no_obs,
                        'epoch':epoch,'seed':seed,'train_record_hash':scaler.fit_rids_hash},ckpt)
        else:
            early+=1
        if valid_dl is not None and early>=cfg.TCN_PATIENCE: break
    logdf=pd.DataFrame(log);logdf.to_csv(RUN/'fit'/f'{tag}_epochs.csv',index=False)
    TCN_HISTORY.extend(log)
    elapsed=time.monotonic()-t0
    del net,train_dl,valid_dl
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
    return {'best_epoch':int(best_epoch),'best_validation_loss':float(best) if validation_indices is not None else None,
            'tag':tag,'duration_sec':elapsed,'checkpoint':str(ckpt)}

def load_tcn_checkpoint(path):
    ckpt=torch.load(path,map_location='cpu',weights_only=False)
    n=CausalTCN(kernel=int(ckpt['kernel']));n.load_state_dict(ckpt['state_dict']);n.eval()
    return n.to(DEVICE)

def tcn_predict(path,indices,scaler,tag='TCN prediction',lookback=48,no_obs=False):
    net=load_tcn_checkpoint(path)
    dl=loader(indices,scaler,training=False,lookback=lookback,no_obs=no_obs)
    out=[]
    with torch.no_grad():
        for seq,hmask,sta,_,_ in tqdm(dl,desc=tag,leave=False):
            y=net(seq.to(DEVICE,non_blocking=True),hmask.to(DEVICE,non_blocking=True),
                  sta.to(DEVICE,non_blocking=True))
            out.append(torch.sigmoid(y).cpu().numpy())
    del net,dl
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
    return np.concatenate(out,axis=0)


## Cell 18 — Fivefold **out-of-fold** XGBoost + TCN training (primary computational stage)

**Per held fold j:** select a 15% **inner stop** subset by whole record among the other four folds; choose XGBoost depth and stop round for I1 and I2 using only inner records; refit each booster on **all four non-held folds**; fit an early-stopped TCN using the same inner record split, then refit it on all four folds for the selected number of epochs with a fresh scaler; generate XGB and TCN interval predictions for every landmark in held fold j, including landmarks whose future I2 is unknown. OOF rows never enter the training or stopping set of the model that scores them.

**Saved artifacts:** candidate tables, selected depths/rounds, two outer boosters per fold, TCN checkpoint and scaler per fold, and a single keyed `oof_predictions.parquet`. Only one unseen prediction exists for each fitting-pool landmark and each base model interval. The per-fold source/timing/event counts accompany the provenance. In full mode this may require substantial Kaggle GPU time; the **pilot** run is designed to catch missing-class, GPU-memory and I/O issues before the definitive frozen run.

In [ ]:
oof_xgb=np.full((len(meta),2),np.nan,dtype=np.float32)
oof_tcn=np.full((len(meta),2),np.nan,dtype=np.float32)
XGB_CANDIDATES=[];XGB_SELECTED=[];TCN_SELECTED=[]
for fold in tqdm(range(cfg.N_FOLDS),desc='Five record-disjoint OOF folds'):
    held=np.flatnonzero((meta.split.to_numpy()=='fit')&(FOLD==fold))
    train_rids=rec.loc[(rec.split=='fit')&(rec.fold!=fold),'rid'].to_numpy()
    held_rids=rec.loc[(rec.split=='fit')&(rec.fold==fold),'rid'].to_numpy()
    inner_rids,stop_rids=grouped_inner_split(train_rids,cfg.SEED+fold)
    fit_idx=ix_by_rids(inner_rids);stop_idx=ix_by_rids(stop_rids)
    outer_idx=ix_by_rids(train_rids)
    assert not set(held_rids)&(set(inner_rids)|set(stop_rids))
    assert not set(inner_rids)&set(stop_rids)
    chosen,logs=select_xgb(fit_idx,stop_idx,cfg.SEED+fold,f'oof{fold}')
    XGB_CANDIDATES.extend(logs)
    for interval,selection in enumerate(chosen):
        booster,info=fit_booster(outer_idx,interval,selection['depth'],selection['rounds'],
                        cfg.SEED+fold,description=f'fold {fold} full-refit I{interval+1}')
        path=RUN/'fit'/f'xgb_oof_fold{fold}_i{interval+1}.json'
        booster.save_model(path)
        oof_xgb[held,interval]=predict_xgb(booster,held)
        XGB_SELECTED.append(dict(selection,fold=fold,refit_path=str(path)))
        del booster
    inner_scaler=SequenceScaler().fit(inner_rids)
    fit_info=tcn_train(fit_idx,inner_scaler,cfg.SEED+fold,f'tcn_fold{fold}_selection',
                       validation_indices=stop_idx)
    final_scaler=SequenceScaler().fit(train_rids)
    final_scaler.dump(RUN/'fit'/f'tcn_oof_fold{fold}_scaler.json')
    full_info=tcn_train(outer_idx,final_scaler,cfg.SEED+fold,f'tcn_fold{fold}_refit',
                        fixed_epochs=fit_info['best_epoch'])
    oof_tcn[held]=tcn_predict(full_info['checkpoint'],held,final_scaler,
                               f'fold {fold} held TCN')
    TCN_SELECTED.append({'fold':fold,'selected_epoch':fit_info['best_epoch'],
                         'validation_loss':fit_info['best_validation_loss'],
                         'train_time_sec':fit_info['duration_sec']+full_info['duration_sec']})
    assert np.isfinite(oof_xgb[held]).all() and np.isfinite(oof_tcn[held]).all()
    json_dump({'fold':fold,'heldout_record_count':len(held_rids),
               'training_record_count':len(train_rids),'inner_fit_count':len(inner_rids),
               'inner_stop_count':len(stop_rids),'xgb_selected':chosen,
               'tcn_selected_epoch':fit_info['best_epoch']},RUN/'fit'/f'fold{fold}_audit.json')

assert np.isfinite(oof_xgb[TRAIN_A]).all() and np.isfinite(oof_tcn[TRAIN_A]).all()
assert np.all(np.isnan(oof_xgb[POLICY_A])) and np.all(np.isnan(oof_tcn[SITE_B]))
oof=meta.loc[TRAIN_A,['site','record','rid','hour','fold','i1','i2','y6','y12']].copy()
for j,name in enumerate(['i1','i2']):
    oof[f'xgb_{name}']=oof_xgb[TRAIN_A,j]
    oof[f'tcn_{name}']=oof_tcn[TRAIN_A,j]
oof.to_parquet(RUN/'scores/oof_predictions.parquet',index=False)
pd.DataFrame(XGB_CANDIDATES).to_csv(RUN/'fit/xgb_candidates.csv',index=False)
pd.DataFrame(XGB_SELECTED).to_csv(RUN/'fit/xgb_selected.csv',index=False)
pd.DataFrame(TCN_SELECTED).to_csv(RUN/'fit/tcn_selected_epochs.csv',index=False)
pd.DataFrame(TCN_HISTORY).to_csv(RUN/'fit/tcn_all_history.csv',index=False)
print('Complete unseen fitting-pool OOF scores:',len(oof),'with no duplicate record-hour keys.')


## Cell 19 — Fit the two interval-specific logistic stack heads and single-model calibrators

**No leakage into the meta learner.** For each interval, select only fitting-pool rows with a known interval target. The first input is `logit(OOF XGBoost probability)`, the second is `logit(OOF TCN probability)`. Fit a two-variable, intercept-included **L2 logistic regression** (`C=1`, `lbfgs`, 1,000 maximum iterations) on those cross-fitted scores. No numeric standardization of logits. Fit two separate one-input logistic models using the **same rows** to calibrate XGBoost and TCN; the calibrated **full-feature XGBoost** is the prespecified strongest simple comparator.

**Stacking is not an additional XGBoost booster.** The trained meta learner contains two coefficients and an intercept per interval. Its coefficients do **not** prove that one feature family causes sepsis; they quantify predictive combination of two OOF signals. Store coefficients, target counts, OOF row-key hash and convergence diagnostics. Complete-case **unknown target masks** differ between intervals, as the protocol requires.

In [ ]:
STACK=[];CAL_XGB=[];CAL_TCN=[];META_AUDIT=[]
for col in (0,1):
    y=ALL_LABELS[TRAIN_A,col]
    known=np.isfinite(y)
    rows=TRAIN_A[known]
    design=np.column_stack([clipped_logit(oof_xgb[rows,col]),clipped_logit(oof_tcn[rows,col])])
    if np.unique(y[known]).size!=2: raise RuntimeError('Meta target has one class')
    fit=LogisticRegression(C=1.0,solver='lbfgs',max_iter=1000,fit_intercept=True)
    fit.fit(design,y[known].astype(int))
    STACK.append(fit)
    cal1=LogisticRegression(C=1.0,solver='lbfgs',max_iter=1000)
    cal1.fit(design[:,[0]],y[known].astype(int));CAL_XGB.append(cal1)
    cal2=LogisticRegression(C=1.0,solver='lbfgs',max_iter=1000)
    cal2.fit(design[:,[1]],y[known].astype(int));CAL_TCN.append(cal2)
    sha=hashlib.sha256(meta.loc[rows,['site','record','hour']].to_csv(index=False).encode()).hexdigest()
    META_AUDIT.append({'interval':col+1,'n_known_targets':len(rows),'n_positive':int(y[known].sum()),
        'key_hash':sha,'stack_intercept':float(fit.intercept_[0]),
        'coef_xgb':float(fit.coef_[0,0]),'coef_tcn':float(fit.coef_[0,1]),
        'xgb_cal_intercept':float(cal1.intercept_[0]),'xgb_cal_coef':float(cal1.coef_[0,0]),
        'tcn_cal_intercept':float(cal2.intercept_[0]),'tcn_cal_coef':float(cal2.coef_[0,0]),
        'stack_iterations':int(fit.n_iter_[0])})

json_dump(META_AUDIT,RUN/'models/meta_calibration.json')
# Save directly interoperable coefficients rather than a dependency on pickled Python objects.
# Every model in this file is exactly recoverable through sigmoid(intercept + X @ coef).
pd.DataFrame(META_AUDIT).to_csv(RUN/'tables/meta_coefficients.csv',index=False)
display(pd.DataFrame(META_AUDIT))


## Cell 20 — Frozen final base estimators, fitted on A fitting pool only

**XGBoost final-selection rule:** For each interval, use the **modal depth** selected by the five OOF fits (ties resolved in favor of shallower depth); use the median selected stopping round among folds that selected this depth, rounded to a positive integer. Refit that exact number of boosting rounds on all A fitting records. Do **not** refit on the policy records after selecting the operating threshold, because that would shift the score distribution at deployment. Save both JSON boosters.

**TCN final-selection rule:** median selected outer-fold epoch, rounded and clamped between 1 and max epochs; fit one 48-hour TCN from scratch on all fitting records with a fresh training-only median/IQR scaler. Preserve its final `state_dict` and scaler; the two interval probabilities share one checkpoint. The stack coefficients from Cell 19 remain frozen and are not refit on A policy or B. This is a *new* run; the code makes no claim to recover historical notebook weights.

In [ ]:
FINAL_XGB=[];FINAL_XGB_CONFIG=[]
for target in (0,1):
    chosen=[x for x in XGB_SELECTED if x['interval']==target+1]
    counts=Counter(item['depth'] for item in chosen)
    mode_depth=sorted(counts,key=lambda depth:(-counts[depth],depth))[0]
    iterations=int(np.floor(np.median([x['rounds'] for x in chosen if x['depth']==mode_depth])+0.5))
    iterations=max(1,iterations)
    booster,fitinfo=fit_booster(TRAIN_A,target,mode_depth,iterations,cfg.SEED,
                      description=f'final XGB interval {target+1}')
    fname=RUN/'models'/f'xgboost_interval_{target+1}.json'
    booster.save_model(fname)
    FINAL_XGB.append(booster)
    FINAL_XGB_CONFIG.append(dict(interval=target+1,depth=mode_depth,rounds=iterations,
                                 artifact=str(fname),**{k:v for k,v in fitinfo.items() if k not in ('depth','rounds','interval')}))
TCN_FINAL_EPOCH=int(np.floor(np.median([x['selected_epoch'] for x in TCN_SELECTED])+0.5))
TCN_FINAL_EPOCH=max(1,TCN_FINAL_EPOCH)
FINAL_SCALER=SequenceScaler().fit(fit_ids)
FINAL_SCALER.dump(RUN/'models/tcn_scaler.json')
tcn_final=tcn_train(TRAIN_A,FINAL_SCALER,cfg.SEED,'tcn_final',fixed_epochs=TCN_FINAL_EPOCH)
shutil.copy2(tcn_final['checkpoint'],RUN/'models/tcn_final.pt')
TCN_FINAL_PATH=RUN/'models/tcn_final.pt'
json_dump({'final_tree':FINAL_XGB_CONFIG,'final_tcn_epoch':TCN_FINAL_EPOCH,
           'final_tcn_params':count_params(CausalTCN()),
           'training_site':'A','training_compartment':'fit only',
           'source_B_untouched_for_training':True},RUN/'models/final_selection.json')
print('Frozen final model configuration:',FINAL_XGB_CONFIG,'TCN epoch',TCN_FINAL_EPOCH)


## Cell 21 — Frozen scoring function; materialize **scores before labels**

**Architecture-to-inference boundary:** For an index array of sample keys, predict interval `a,b` using each of the two XGB JSON boosters and the shared TCN checkpoint; apply the two fitted logistic meta heads to the corresponding clipped logits. Return six methods: uncalibrated XGB, calibrated XGB, raw TCN, calibrated TCN, fixed 50:50 *interval probability* mean, and the fitted heterogeneous stack. Convert each interval pair to cumulative P6/P12 using `P12=a+(1−a)b`. No code inside this function reads labels, event times, record endpoints or site. Scoring is batched with `tqdm` for GPU inference.

**Prespecified comparison:** The six-hour AP of `stack` minus `calibrated_xgb` on identical observed labels and record-hour keys is the primary paired contrast. Tree-alone vs. stack, TCN alone vs. stack, and equal-average vs. fitted stack are **architectural ablations without additional fits** and therefore cheap enough to include in every run. Write A policy and site-B prediction tables to Parquet **without outcomes** before the policy/evaluation cells join them.

In [ ]:
METHODS=['raw_xgb','calibrated_xgb','raw_tcn','calibrated_tcn','equal_average','stack']
HORIZON_NAMES=['p6','p12']
def predict_methods(indices,tag):
    xgb_inter=np.column_stack([predict_xgb(m,indices) for m in FINAL_XGB])
    tcn_inter=tcn_predict(TCN_FINAL_PATH,indices,FINAL_SCALER,tag+' TCN')
    vals={'raw_xgb':xgb_inter,'raw_tcn':tcn_inter,
          'equal_average':0.5*xgb_inter+0.5*tcn_inter}
    stacked=[];tree_cal=[];tcn_cal=[]
    for interval in (0,1):
        a=clipped_logit(xgb_inter[:,interval]);b=clipped_logit(tcn_inter[:,interval])
        stacked.append(STACK[interval].predict_proba(np.column_stack([a,b]))[:,1])
        tree_cal.append(CAL_XGB[interval].predict_proba(a[:,None])[:,1])
        tcn_cal.append(CAL_TCN[interval].predict_proba(b[:,None])[:,1])
    vals['stack']=np.column_stack(stacked)
    vals['calibrated_xgb']=np.column_stack(tree_cal)
    vals['calibrated_tcn']=np.column_stack(tcn_cal)
    table=meta.loc[indices,['rid','site','record','hour']].reset_index(drop=True).copy()
    for method,interval in vals.items():
        p=compose_intervals(interval)
        table[f'{method}_p6']=p[:,0].astype(np.float32)
        table[f'{method}_p12']=p[:,1].astype(np.float32)
    for method in METHODS:
        assert np.isfinite(table[[f'{method}_p6',f'{method}_p12']].to_numpy()).all()
    return table

policy_scores=predict_methods(POLICY_A,'A policy')
policy_scores.to_parquet(RUN/'scores/policy_scores_before_outcome_join.parquet',index=False)
# Important: this table is generated once, using the same final fitted models as A policy.
b_scores=predict_methods(SITE_B,'Untouched B')
b_scores.to_parquet(RUN/'scores/source_B_scores_before_outcome_join.parquet',index=False)
assert not policy_scores.duplicated(['site','record','hour']).any()
assert not b_scores.duplicated(['site','record','hour']).any()
print('Frozen outcome-free predictions saved:',len(policy_scores),'A-policy and',len(b_scores),'B samples')


## Cell 22 — Securely join frozen prediction keys to the independent outcome registry

**Label separation:** The score tables were already written in Cell 21. The present cell performs a **strict one-to-one key join** with event occurrence, known-observation endpoint and `y6/y12`. Missing matches are fatal; a join of predictions to future labels without asserting key cardinality can yield apparently excellent but invalid AP. The downstream evaluation code never alters the frozen model or operating thresholds.

**Risk-set denominators:** Predictions exist even for hours without full future follow-up, but these unknown targets are excluded from horizon-specific classification losses. They remain in the alarm replay so issued alerts can be marked **indeterminate**. Maintain both `n_monitored_hours` and `n_evaluable_hours` to avoid misreporting false alarms.

In [ ]:
label_cols=['site','record','hour','rid','event_hour','last_good','y6','y12','i1','i2']
labels=meta[label_cols]
def strict_join(pred,labels):
    q=pred.merge(labels,on=['site','record','hour','rid'],how='left',validate='one_to_one',indicator=True)
    if (q['_merge']!='both').any() or len(q)!=len(pred): raise AssertionError('Prediction/target keys differ')
    return q.drop(columns='_merge').sort_values(['rid','hour']).reset_index(drop=True)
policy_eval=strict_join(policy_scores,labels)
b_eval=strict_join(b_scores,labels)
assert policy_eval.site.eq('A').all() and b_eval.site.eq('B').all()
assert b_eval[['rid','hour']].duplicated().sum()==0
print('A-policy known targets:',policy_eval[['y6','y12']].count().to_dict())
print('B known targets:',b_eval[['y6','y12']].count().to_dict())


## Cell 23 — A-only threshold rule and deterministic six-hour alert cooldown

A clinical risk score and an alert episode are **not the same object**. This replay emits an alert at hour t only if the method's probability passes the chosen threshold and no alert was emitted within the preceding 6 hours for that record. A true episode means the **first observed label transition** is in `(t,t+H]`. A confirmed false episode requires continuous known-negative follow-up through `t+H`; an episode lacking both a qualifying event and complete follow-up is **indeterminate**. Count the underlying event **only once**, regardless of repeated alert episodes.

**Frozen A policy:** sweep all unique score quantiles at 0%,1%,...,100%, plus threshold >1 for a zero-alert point. Among those with at most **2 confirmed false episodes per 100 evaluable hours**, maximize **fraction of incident records detected**. Break ties by fewer emitted alerts, then by higher threshold. This is an *illustrative research budget*, not a clinical deployment guideline. Each method and each horizon gets a separately selected A-only threshold. Apply exactly these thresholds to B even if its false-alert burden differs.

In [ ]:
@njit(cache=False)
def replay_numba(hours,offsets,scores,event_hours,last_good,threshold,horizon,cooldown,known_evaluable):
    # All inputs are sorted by record and hour. offsets: first row of each record plus terminal length.
    nrec=len(offsets)-1
    emitted=0;true=0;false=0;indeterminate=0;detected=0;monitored=len(hours)
    evaluable=0;events=0;lead_sum=0.;eligible_events=0
    leads=np.empty(nrec,dtype=np.float32);leads[:]=np.nan
    for r in range(nrec):
        lo=offsets[r];hi=offsets[r+1]
        last_alert=-10**9;was_detected=False;first_lead=np.nan
        event=event_hours[r];end=last_good[r]
        if event>=0:
            events+=1
            if hi>lo: eligible_events+=1
        for j in range(lo,hi):
            if known_evaluable[j]: evaluable+=1
            t=hours[j]
            if scores[j]>=threshold and t>=last_alert+cooldown:
                emitted+=1;last_alert=t
                if event>=0 and event>t and event<=t+horizon:
                    true+=1;was_detected=True
                    lead=event-t
                    if np.isnan(first_lead) or lead>first_lead: first_lead=lead
                elif end>=t+horizon: false+=1
                else: indeterminate+=1
        if was_detected:
            detected+=1;leads[r]=first_lead
    return (emitted,true,false,indeterminate,detected,monitored,evaluable,
            events,eligible_events,leads)

def prepare_replay(q,horizon):
    q=q.sort_values(['rid','hour']).reset_index(drop=True)
    ids=q.rid.to_numpy(dtype=np.int32)
    starts=np.r_[0,np.flatnonzero(np.diff(ids)!=0)+1,len(q)].astype(np.int64)
    events=q.event_hour.to_numpy(dtype=np.int32)[starts[:-1]]
    last=q.last_good.to_numpy(dtype=np.int32)[starts[:-1]]
    known=np.isfinite(q[f'y{horizon}'].to_numpy())
    return {'q':q,'hours':q.hour.to_numpy(dtype=np.int32),'starts':starts,
            'events':events,'last':last,'known':known}

def replay_prepared(p,col,threshold,horizon):
    scores=p['q'][col].to_numpy(dtype=np.float32)
    z=replay_numba(p['hours'],p['starts'],scores,p['events'],p['last'],float(threshold),
                   int(horizon),int(cfg.ALERT_COOLDOWN_HOURS),p['known'])
    emitted,true,false,unknown,detected,monitor,evalhours,events,eligible,leads=z
    return {'threshold':float(threshold),'emitted_episodes':int(emitted),
            'true_episodes':int(true),'false_episodes':int(false),
            'indeterminate_episodes':int(unknown),'detected_event_records':int(detected),
            'eligible_event_records':int(eligible),'all_incident_events':int(events),
            'monitored_hours':int(monitor),'evaluable_hours':int(evalhours),
            'event_sensitivity':float(detected/eligible) if eligible else np.nan,
            'false_episodes_per_100_evaluable_h':float(100*false/evalhours) if evalhours else np.nan,
            'all_episodes_per_100_monitored_h':float(100*emitted/monitor) if monitor else np.nan,
            'episode_ppv':float(true/(true+false)) if (true+false) else np.nan,
            'lead_time_median_h':float(np.nanmedian(leads)) if detected else np.nan,
            'lead_time_q25_h':float(np.nanquantile(leads,.25)) if detected else np.nan,
            'lead_time_q75_h':float(np.nanquantile(leads,.75)) if detected else np.nan}

# Time-boundary fixture with a single event at hour 16 and an alert emitted at hour 10.
fixture_q=pd.DataFrame({'rid':[0]*7,'hour':list(range(10,17)),'event_hour':[16]*7,
    'last_good':[16]*7,'y6':[1.,1.,1.,1.,1.,1.,np.nan],
    'stack_p6':[.9,.9,.9,.9,.9,.9,0.]})
fixture=prepare_replay(fixture_q,6)
res=replay_prepared(fixture,'stack_p6',.5,6)
assert res['emitted_episodes']==1 and res['true_episodes']==1 and res['detected_event_records']==1
assert replay_prepared(fixture,'stack_p6',1.1,6)['emitted_episodes']==0
print('Six-hour cooldown, exact event boundary and zero-alert policy fixtures PASS.')


## Cell 24 — Select thresholds on A policy (no B outcomes inspected)

Build a deterministic quantile grid *separately for every method/horizon* using A-policy scores. The policy function replays all A records at each threshold, reports confirmed false alerts per 100 **evaluable** monitored hours and event sensitivity among records with score opportunities. Record indeterminate episode counts, total episodes per 100 monitored hours and episode PPV. This is a crucial distinction from the original paper's simple classification threshold: an hour-level false positive can be suppressed by cooldown, while one event may have multiple risk hours.

**Output:** a `thresholds.json`, detailed operating-point CSV and full A-only sensitivity–burden curves. If the budget cannot be met except by the zero-alert threshold, the frozen threshold remains zero-alert; **never** loosen the budget with information from B.

In [ ]:
POLICY_THRESHOLDS={};POLICY_CURVES=[];POLICY_SELECTED=[]
for horizon in cfg.HORIZONS:
    prepared=prepare_replay(policy_eval,horizon)
    for method in tqdm(METHODS,desc=f'Select {horizon}h alarm policies'):
        col=f'{method}_p{horizon}'
        scores=prepared['q'][col].to_numpy()
        grid=np.unique(np.r_[np.quantile(scores,np.linspace(0,1,101)),1.0000001])
        candidates=[]
        for threshold in tqdm(grid,desc=f'{method} {horizon}h quantiles',leave=False):
            result=replay_prepared(prepared,col,threshold,horizon)
            result.update(model=method,horizon=horizon,cohort='A_policy')
            candidates.append(result)
        POLICY_CURVES.extend(candidates)
        feasible=[x for x in candidates if np.isfinite(x['false_episodes_per_100_evaluable_h']) and
                  x['false_episodes_per_100_evaluable_h']<=cfg.ALERT_FALSE_BUDGET+1e-12]
        if not feasible: raise RuntimeError('No feasible threshold including zero-alert; check denominators')
        best=sorted(feasible,key=lambda q:(-q['event_sensitivity'] if np.isfinite(q['event_sensitivity']) else 0,
                                           q['emitted_episodes'],-q['threshold']))[0]
        POLICY_THRESHOLDS[f'{method}_{horizon}']=best['threshold']
        POLICY_SELECTED.append(best)

pd.DataFrame(POLICY_CURVES).to_csv(RUN/'policy/a_policy_all_quantiles.csv',index=False)
pd.DataFrame(POLICY_SELECTED).to_csv(RUN/'policy/a_frozen_operating_points.csv',index=False)
json_dump({'thresholds':POLICY_THRESHOLDS,'budget_confirmed_false_per_100_evaluable_h':cfg.ALERT_FALSE_BUDGET,
           'cooldown_h':cfg.ALERT_COOLDOWN_HOURS,'source':'A policy only',
           'quantile_grid':'0,1,...,100 percentiles + 1.0000001'},RUN/'policy/thresholds.json')
display(pd.DataFrame(POLICY_SELECTED)[['model','horizon','threshold','event_sensitivity',
                                      'false_episodes_per_100_evaluable_h','indeterminate_episodes']])


## Cell 25 — Publishable landmark metrics on the frozen B predictions

This step reads only the previously saved **frozen site-B predictions**, joins them to the known future labels, and computes the endpoints. For each horizon and method report `n_evaluable`, `n_positive`, natural positive prevalence, **average precision (non-trapezoidal)**, AUROC, Brier score and binary log loss. A 6-hour score is never scored on an unknown 6-hour label; a 12-hour score is never scored on an unknown 12-hour label. Include the development-only prevalence-only prediction as a calibration reference, and an equal-record-weighted sensitivity to show whether long records dominate the hour-weighted results.

**Research interpretation:** AP depends on prevalence and the landmark unit, so it must not be compared numerically with an earlier terminal-control AP as if the datasets were equivalent. This experiment directly tests whether adding the ordered TCN representation improves the strong calibrated 512-feature XGBoost, using identical B keys. A marginally higher AP without meaningful alarm gain is not evidence of clinical benefit. Figures and CIs appear in subsequent cells.

In [ ]:
def metrics_table(q,methods=METHODS):
    output=[]
    for h in cfg.HORIZONS:
        good=np.isfinite(q[f'y{h}'].to_numpy())
        yt=q.loc[good,f'y{h}'].to_numpy(dtype=int)
        ids=q.loc[good,'rid'].to_numpy(dtype=np.int32)
        _,inverse,cnt=np.unique(ids,return_inverse=True,return_counts=True)
        weights=1/cnt[inverse].astype(np.float64)
        for method in methods:
            col=f'{method}_p{h}'
            if col not in q.columns:continue
            p=q.loc[good,col].to_numpy(dtype=np.float64)
            if len(yt)==0 or len(np.unique(yt))<2:continue
            output.append({'site':str(q.site.iloc[0]),'method':method,'horizon_h':h,
                'n_evaluable_landmarks':len(yt),'n_positive_landmarks':int(yt.sum()),
                'positive_landmark_prevalence':float(yt.mean()),
                'n_distinct_records':int(len(cnt)),
                'average_precision':float(average_precision_score(yt,p)),
                'equal_record_weighted_ap':float(average_precision_score(yt,p,sample_weight=weights)),
                'AUROC':float(roc_auc_score(yt,p)),
                'Brier':float(brier_score_loss(yt,p)),
                'log_loss':float(log_loss(yt,np.clip(p,1e-6,1-1e-6),labels=[0,1]))})
    return pd.DataFrame(output)

RESULTS=metrics_table(b_eval)
RESULTS.to_csv(RUN/'tables/held_site_B_main_metrics.csv',index=False)
display(RESULTS[['method','horizon_h','average_precision','AUROC','Brier','log_loss',
                 'n_evaluable_landmarks','positive_landmark_prevalence']])
# A-only baseline prevalence is computed on known cumulative outcomes in the fitting pool.
prevalence_dev={h:float(meta.loc[(meta.split=='fit')&meta[f'y{h}'].notna(),f'y{h}'].mean())
                for h in cfg.HORIZONS}
prevalence_baselines=[]
for h in cfg.HORIZONS:
    good=b_eval[f'y{h}'].notna(); yt=b_eval.loc[good,f'y{h}'].to_numpy(dtype=int)
    p=np.full(len(yt),prevalence_dev[h])
    prevalence_baselines.append({'horizon':h,'A_fit_prevalence':prevalence_dev[h],
         'B_AP_baseline':average_precision_score(yt,p),
         'B_Brier_baseline':brier_score_loss(yt,p),
         'B_log_loss_baseline':log_loss(yt,np.clip(p,1e-6,1-1e-6),labels=[0,1])})
json_dump(prevalence_baselines,RUN/'tables/prevalence_only_baseline.json')
assert all((b_eval[f'{m}_p6'].to_numpy()<=b_eval[f'{m}_p12'].to_numpy()+1e-7).all() for m in METHODS)


## Cell 26 — Evaluate frozen alert thresholds on B and count lead times

Use **exactly** the thresholds from Cell 24; there is no reoptimization on B. The alert replay returns total emitted episodes, confirmed true/false episodes, **indeterminate episodes**, events detected out of incident records with an opportunity, the all-incident-event count, median and interquartile range of first qualifying alert-to-label-transition time, 100-hour false burden and total alert burden. An alert at hour 10 prevents repeats at hours 11–15; the next possible alert hour is 16. A patient with many qualifying hours can still count as one detected event.

**Denominator caveat:** false alarm rate divides by *evaluable monitored hours*, while total alert rate divides by *all monitored hours*. These are different denominators and must be labeled explicitly in the paper. Indeterminate episodes appear separately and should not be forced into the false category. A clinical alarm utility claim requires a better outcome than the benchmark label, external event timestamps and a prospective study.

In [ ]:
B_ALERT_ROWS=[]
for h in cfg.HORIZONS:
    prepared=prepare_replay(b_eval,h)
    for method in tqdm(METHODS,desc=f'Frozen B {h}h alarm evaluation'):
        t=POLICY_THRESHOLDS[f'{method}_{h}']
        out=replay_prepared(prepared,f'{method}_p{h}',t,h)
        out.update(model=method,horizon=h,site='B',threshold_origin='A_policy')
        B_ALERT_ROWS.append(out)
ALERT_RESULTS=pd.DataFrame(B_ALERT_ROWS)
ALERT_RESULTS.to_csv(RUN/'tables/held_site_B_frozen_alerts.csv',index=False)
display(ALERT_RESULTS[['model','horizon','threshold','event_sensitivity',
   'false_episodes_per_100_evaluable_h','all_episodes_per_100_monitored_h',
   'indeterminate_episodes','lead_time_median_h']])


## Cell 27 — Paired, record-clustered AP bootstrap with exact ties

**Primary statistical contrast:** site-B six-hour AP of **stack − calibrated full-feature XGBoost**, sampled as **entire records** with replacement, not independent hours. A record's predictions and all positive/negative landmarks move together. Compute AP with multiplicity weights, including **exact-score ties** grouped into one threshold; this agrees with sklearn's non-interpolated AP at the original sample. Reuse the same bootstrap patient draw for both methods. Report the point difference, 95% percentile interval and the number of undefined replicates (e.g. no positives).

**Efficiency:** sorting a million scores on each of 2,000 replicates would be slow; sort each model **once** and then apply record multiplicities as sample weights. Primary full protocol defaults to **2,000** resamples; in pilot it is capped at 200 and clearly labeled as a debugging estimate. This bootstrap conditions on fixed fitted models and the B site; it does **not** measure uncertainty over possible hospitals or random training seeds.

In [ ]:
class WeightedAPSorted:
    def __init__(self,y,scores,record_ids):
        order=np.argsort(-np.asarray(scores),kind='mergesort')
        s=np.asarray(scores)[order]
        self.y=np.asarray(y,dtype=np.float64)[order]
        self.ridx=np.asarray(record_ids,dtype=np.int32)[order]
        self.ends=np.r_[np.flatnonzero(np.diff(s)!=0),len(s)-1]
    def score(self,record_multiplicity):
        w=record_multiplicity[self.ridx]
        pos=w*self.y
        cp=np.cumsum(pos,dtype=np.float64)[self.ends]
        cw=np.cumsum(w,dtype=np.float64)[self.ends]
        tot=cp[-1]
        if tot==0 or (cw[-1]-tot)==0: return np.nan
        incr=np.diff(np.r_[0.,cp]); prec=np.divide(cp,cw,out=np.zeros_like(cp),where=cw>0)
        return float(np.dot(prec,incr)/tot)

def paired_ap_bootstrap(q,first,second,horizon,n_boot,seed,desc='Paired record bootstrap'):
    good=q[f'y{horizon}'].notna()
    sub=q.loc[good,['rid',f'y{horizon}',f'{first}_p{horizon}',f'{second}_p{horizon}']]
    rids,inverse=np.unique(sub.rid.to_numpy(),return_inverse=True)
    y=sub[f'y{horizon}'].to_numpy(dtype=np.int8)
    left=WeightedAPSorted(y,sub[f'{first}_p{horizon}'].to_numpy(),inverse)
    right=WeightedAPSorted(y,sub[f'{second}_p{horizon}'].to_numpy(),inverse)
    mult=np.ones(len(rids),dtype=np.float64)
    # Verify weighted exact-tie AP against sklearn before resampling.
    assert abs(left.score(mult)-average_precision_score(y,sub[f'{first}_p{horizon}']))<1e-10
    assert abs(right.score(mult)-average_precision_score(y,sub[f'{second}_p{horizon}']))<1e-10
    rng=np.random.default_rng(seed)
    diffs=[];undefined=0
    for _ in tqdm(range(n_boot),desc=desc):
        draw=rng.integers(0,len(rids),size=len(rids))
        multiplicity=np.bincount(draw,minlength=len(rids)).astype(np.float64)
        a=left.score(multiplicity);b=right.score(multiplicity)
        if not(np.isfinite(a) and np.isfinite(b)):
            undefined+=1;continue
        diffs.append(a-b)
    a0=left.score(mult);b0=right.score(mult)
    return {'model_a':first,'model_b':second,'horizon':horizon,'n_boot':n_boot,
            'n_defined':len(diffs),'n_undefined':undefined,
            'AP_a':a0,'AP_b':b0,'AP_difference':a0-b0,
            'CI_low':float(np.percentile(diffs,2.5)) if len(diffs) else np.nan,
            'CI_high':float(np.percentile(diffs,97.5)) if len(diffs) else np.nan},np.asarray(diffs)

boot_count=cfg.BOOTSTRAPS if cfg.RUN_MODE=='full' else min(200,cfg.BOOTSTRAPS)
PRIMARY_CI,primary_draws=paired_ap_bootstrap(b_eval,'stack','calibrated_xgb',6,boot_count,
     cfg.SEED,'Primary B 6h paired bootstrap')
json_dump(PRIMARY_CI,RUN/'tables/primary_AP_bootstrap_CI.json')
pd.DataFrame({'paired_AP_difference':primary_draws}).to_csv(RUN/'tables/primary_AP_bootstrap_draws.csv',index=False)
print('PRIMARY STACK − CALIBRATED XGB, 6h B AP:',PRIMARY_CI)


## Cell 28 — Journal figures: precision–recall, ROC and cumulative horizon validity

Produce per-horizon **precision–recall (PR)** comparisons on the **same** B patient-hour keys and natural prevalence, with a horizontal prevalence-only reference. PR is prioritized over ROC because a useful operating precision can be difficult to interpret from AUROC in rare-event settings. Add supplemental ROC plots, showing that it answers a different question than event-level alarm replay. The main article can use the stack, calibrated XGBoost and TCN curves; the supplement includes all six methods. Save PDF and SVG for vector reproduction and PNG at 220 DPI for review.

Additionally plot `P6` vs `P12` on a deterministic sample of B rows; every point must satisfy `P12≥P6`. This is an architecture invariant rather than a performance metric. Both horizon plots use unchanged scores; no threshold is tuned on B.

In [ ]:
COLORS={'raw_xgb':'#777777','calibrated_xgb':'#1565c0','raw_tcn':'#ed6c02',
        'calibrated_tcn':'#b35d00','equal_average':'#009688','stack':'#9334e6'}
for h in cfg.HORIZONS:
    mask=b_eval[f'y{h}'].notna()
    yt=b_eval.loc[mask,f'y{h}'].to_numpy(dtype=int)
    plt.figure(figsize=(8,6))
    for method in METHODS:
        p=b_eval.loc[mask,f'{method}_p{h}'].to_numpy()
        precision,recall,_=precision_recall_curve(yt,p)
        ap=average_precision_score(yt,p)
        plt.plot(recall,precision,label=f'{method} AP={ap:.3f}',linewidth=1.5,color=COLORS[method])
    plt.axhline(yt.mean(),linestyle='--',color='black',label=f'Prevalence={yt.mean():.3f}')
    plt.xlabel('Recall at known landmark-hours');plt.ylabel('Precision')
    plt.title(f'Held-out B precision–recall, {h}h label-transition window')
    plt.legend(fontsize=8);plt.grid(alpha=.2);save_fig(f'03_PR_{h}h')
    plt.figure(figsize=(7,6))
    for method in METHODS:
        p=b_eval.loc[mask,f'{method}_p{h}'].to_numpy()
        fpr,tpr,_=roc_curve(yt,p)
        plt.plot(fpr,tpr,label=f'{method} AUC={roc_auc_score(yt,p):.3f}',color=COLORS[method])
    plt.plot([0,1],[0,1],'k--',label='Chance')
    plt.xlabel('False-positive rate');plt.ylabel('True-positive rate')
    plt.title(f'Held-out B ROC, {h}h');plt.legend(fontsize=8);save_fig(f'04_ROC_{h}h')

rng=np.random.default_rng(cfg.SEED)
choose=rng.choice(len(b_eval),size=min(15000,len(b_eval)),replace=False)
plt.figure(figsize=(7,6))
plt.scatter(b_eval.loc[choose,'stack_p6'],b_eval.loc[choose,'stack_p12'],s=3,alpha=.35)
plt.plot([0,1],[0,1],'k--')
plt.xlim(0,1);plt.ylim(0,1);plt.xlabel('Stack P(transition by 6 hours)')
plt.ylabel('Stack P(transition by 12 hours)')
plt.title('Architectural coherence check: all points on/above diagonal')
save_fig('05_probability_horizon_coherence')


## Cell 29 — Reliability diagrams, calibration coefficients and bin denominators

A model can rank event hours well while emitting **miscalibrated probabilities** at another hospital. For each method and horizon, assign held-out B predictions into fixed `[0,0.1,...,1]` bins, compute mean predicted risk, observed event rate and sample count. Plot the diagonal “perfect calibration” line. Fit a descriptive (not deployed) calibration intercept and slope on B and record them **as evaluation metrics only**; the evaluation coefficients do **not** feed back into the model. Use a one-input model on `logit(p)` for slope and an intercept-only correction with logit as an offset for calibration-in-the-large.

**Why bins count?** A small high-risk bin may contain too few events for precise estimated reliability; printing its denominator prevents over-interpreting jagged plots. Report Brier and log loss from the same known subset. Site B is not used for any new recalibration—this figure measures transport, not a repair to it.

In [ ]:
def calibration_intercept_slope(y,p):
    p=np.clip(np.asarray(p),1e-6,1-1e-6)
    x=logit(p)
    # Calibration slope: descriptive regression, not a second-site fitted deployment model.
    lr=LogisticRegression(C=1e6,solver='lbfgs',max_iter=1000)
    lr.fit(x.reshape(-1,1),y)
    # Offset intercept: solve sum(sigmoid(logit(p)+c)-y)=0, with a stable bisection.
    lo,hi=-30.,30.
    for _ in range(80):
        mid=(lo+hi)/2
        if expit(x+mid).mean()>np.mean(y):hi=mid
        else:lo=mid
    return float((lo+hi)/2),float(lr.coef_[0,0]),float(lr.intercept_[0])

CAL_RESULTS=[];BIN_ROWS=[]
for h in cfg.HORIZONS:
    known=b_eval[f'y{h}'].notna()
    yt=b_eval.loc[known,f'y{h}'].to_numpy(dtype=int)
    plt.figure(figsize=(7.5,6.5))
    for method in METHODS:
        p=b_eval.loc[known,f'{method}_p{h}'].to_numpy()
        c_int,c_slope,c_intercept=calibration_intercept_slope(yt,p)
        CAL_RESULTS.append({'model':method,'horizon':h,'calibration_in_large_intercept':c_int,
                           'calibration_slope':c_slope,'logistic_intercept':c_intercept,
                           'n_evaluable':len(yt)})
        bin_id=np.clip(np.digitize(p,np.linspace(0,1,11),right=False)-1,0,9)
        bx=[];by=[]
        for b in range(10):
            ix=bin_id==b
            BIN_ROWS.append({'model':method,'horizon':h,'bin':b,'lower':b/10,'upper':(b+1)/10,
                             'n':int(ix.sum()),'mean_prediction':float(np.mean(p[ix])) if ix.any() else np.nan,
                             'observed_frequency':float(np.mean(yt[ix])) if ix.any() else np.nan})
            if ix.sum()>0: bx.append(np.mean(p[ix]));by.append(np.mean(yt[ix]))
        plt.plot(bx,by,marker='o',markersize=3,label=method,color=COLORS[method])
    plt.plot([0,1],[0,1],'k--',label='Ideal calibration')
    plt.xlabel('Mean frozen predicted risk (fixed 0.1 bins)')
    plt.ylabel('Observed positive-label frequency')
    plt.title(f'Held-out B reliability diagram, {h}h')
    plt.legend(fontsize=8);save_fig(f'06_reliability_{h}h')
pd.DataFrame(CAL_RESULTS).to_csv(RUN/'tables/held_site_B_calibration_parameters.csv',index=False)
pd.DataFrame(BIN_ROWS).to_csv(RUN/'tables/held_site_B_calibration_bin_counts.csv',index=False)


## Cell 30 — Development-frozen threshold curves displayed on B

The research policy is chosen in A only, but it is informative to **display** what event detection and false episode burden look like at a family of **A-specified thresholds** when applied to B. Use the already determined A 0–100% quantile thresholds for each model/horizon; this is descriptive plotting, **not** a fresh threshold-selection procedure. Mark the exact A-selected operating point on the B graph. Include a zero-alert point. Save both the continuous curve data and frozen B operating-point table.

**Data semantics:** a point shows **detected incident records / eligible event records** versus **confirmed false emitted alert episodes per 100 evaluable monitored hours**, after the six-hour cooldown. This is more operationally meaningful than hour-level sensitivity at threshold 0.5. Differences between A's target budget and B's realized burden reflect site transport and cannot be masked by retuning B.

In [ ]:
curves=pd.DataFrame(POLICY_CURVES)
EXTERNAL_CURVES=[]
for h in cfg.HORIZONS:
    prepared=prepare_replay(b_eval,h)
    for method in tqdm(METHODS,desc=f'B descriptive burden curves {h}h'):
        th=curves[(curves.model==method)&(curves.horizon==h)]['threshold'].unique()
        for threshold in th:
            row=replay_prepared(prepared,f'{method}_p{h}',float(threshold),h)
            row.update(model=method,horizon=h,site='B',role='descriptive—not site-B threshold fitting')
            EXTERNAL_CURVES.append(row)

external=pd.DataFrame(EXTERNAL_CURVES)
external.to_csv(RUN/'tables/B_descriptive_alarm_curve.csv',index=False)
for h in cfg.HORIZONS:
    plt.figure(figsize=(8,6))
    for method in METHODS:
        subset=external[(external.horizon==h)&(external.model==method)].sort_values('false_episodes_per_100_evaluable_h')
        plt.plot(subset['false_episodes_per_100_evaluable_h'],subset['event_sensitivity'],
                 label=method,alpha=.85,color=COLORS[method])
        selection=ALERT_RESULTS[(ALERT_RESULTS.horizon==h)&(ALERT_RESULTS.model==method)].iloc[0]
        plt.scatter([selection['false_episodes_per_100_evaluable_h']],
                    [selection['event_sensitivity']],s=38,color=COLORS[method],edgecolors='black')
    plt.xlabel('Confirmed false alert episodes / 100 evaluable monitored hours')
    plt.ylabel('Fraction of incident events detected')
    plt.title(f'B alert-burden tradeoff, {h}h; dots = frozen A thresholds')
    plt.grid(alpha=.2);plt.legend(fontsize=8);save_fig(f'07_alarm_burden_{h}h')


## Cell 31 — Development-trained tabular representation ablations

The primary tree gets all 512 fields. Additional **independently trained tree models** isolate the effect of individual representations: **state-only** (`104` features), **trajectory-only** (`408`), **observation-process-only** (`measured_now`, `age_hours`, window counts), and **no explicit observation fields** (remove flags, ages and measurement counts). These are retrained on A's fitting pool with the same depth grid and inner record-grouped stopping logic; applying the full model to masked columns without refitting would not be a valid ablation.

**Exact contrasts:** State-only tests current physiology versus full historical summaries; trend-only tests history without current-state features; observation-only tests sampling practice as a predictor; no-explicit-observation tests how much the tree relies on explicit measurement practices while noting that NaN routing still encodes some missingness. These are **exploratory secondary B comparisons**, not substitutes for the main stack-vs-calibrated-tree paired endpoint. Optional `RUN_TREE_ABLATIONS=False` skips these additional fits and explicitly records that absence in the report.

In [ ]:
obs_cols=[c for c in FEATURE_COLS if c.endswith('__measured_now') or
          c.endswith('__age_hours') or c.endswith('__count')]
no_obs_cols=[c for c in FEATURE_COLS if c not in obs_cols]
ABLATION_COLS={'state_only':STATE_COLS,'trajectory_only':TRAJ_COLS,
               'observation_only':obs_cols,'without_explicit_observation':no_obs_cols}
assert len(ABLATION_COLS['state_only'])==104 and len(ABLATION_COLS['trajectory_only'])==408
assert len(ABLATION_COLS['observation_only'])==136
ABLATION_RESULTS=[];ABLATION_FITS=[]

def train_subset_booster(cols,train_idx,stop_idx,target,depth,rounds,seed,tag):
    ci=np.array([FEATURE_COLS.index(c) for c in cols],dtype=np.int32)
    ya=ALL_LABELS[train_idx,target];good=np.isfinite(ya)
    tr=train_idx[good];y=ya[good]
    dtr=xgb.DMatrix(X[np.ix_(tr,ci)],label=y,missing=np.nan,feature_names=list(cols))
    evals=[]
    if stop_idx is not None:
        ys=ALL_LABELS[stop_idx,target];ok=np.isfinite(ys)
        if np.unique(ys[ok]).size!=2:raise RuntimeError(f'Ablation {tag}: one-class stop set')
        evals=[(xgb.DMatrix(X[np.ix_(stop_idx[ok],ci)],label=ys[ok],feature_names=list(cols)), 'stop')]
    p=xgb_params(depth,seed)
    progress=XGBProgress(tag,int(rounds))
    try:
        model=xgb.train(p,dtr,num_boost_round=int(rounds),evals=evals,
            early_stopping_rounds=cfg.XGB_PATIENCE if evals else None,
            callbacks=[progress],verbose_eval=False)
    finally:progress.close()
    best_round=int(model.best_iteration+1) if evals else int(rounds)
    best_loss=float(model.best_score) if evals else np.nan
    return model,best_round,best_loss

def subset_predict(booster,indices,cols):
    ci=np.array([FEATURE_COLS.index(c) for c in cols],dtype=np.int32)
    pred=np.empty(len(indices),dtype=np.float32)
    for start in range(0,len(indices),250000):
        ix=indices[start:start+250000]
        pred[start:start+len(ix)]=booster.predict(xgb.DMatrix(X[np.ix_(ix,ci)],feature_names=list(cols),missing=np.nan))
    return pred

if cfg.RUN_TREE_ABLATIONS:
    abl_train_rids,abl_stop_rids=grouped_inner_split(fit_ids,cfg.SEED+77)
    abl_train=ix_by_rids(abl_train_rids);abl_stop=ix_by_rids(abl_stop_rids)
    for name,cols in tqdm(ABLATION_COLS.items(),desc='Tree representation ablations'):
        prob_policy=[];prob_B=[]
        for j in (0,1):
            trials=[]
            for depth in cfg.XGB_DEPTHS:
                booster,r,loss=train_subset_booster(cols,abl_train,abl_stop,j,depth,
                    cfg.XGB_ROUNDS,cfg.SEED+77,f'{name} I{j+1} d{depth}')
                trials.append({'depth':depth,'rounds':r,'valid_logloss':loss})
                del booster
            chosen=sorted(trials,key=lambda z:(z['valid_logloss'],z['depth'],z['rounds']))[0]
            booster,_,_=train_subset_booster(cols,TRAIN_A,None,j,chosen['depth'],
                chosen['rounds'],cfg.SEED+77,f'{name} final I{j+1}')
            booster.save_model(RUN/'fit'/f'ablation_{name}_i{j+1}.json')
            prob_policy.append(subset_predict(booster,POLICY_A,cols))
            prob_B.append(subset_predict(booster,SITE_B,cols))
            ABLATION_FITS.append({'name':name,'interval':j+1,'n_features':len(cols),**chosen})
            del booster
        p=compose_intervals(np.column_stack(prob_B))
        b_eval[f'{name}_p6']=p[:,0];b_eval[f'{name}_p12']=p[:,1]
        policy_p=compose_intervals(np.column_stack(prob_policy))
        policy_eval[f'{name}_p6']=policy_p[:,0];policy_eval[f'{name}_p12']=policy_p[:,1]
    ABLATION_RESULTS.extend(metrics_table(b_eval,list(ABLATION_COLS)).to_dict(orient='records'))
    pd.DataFrame(ABLATION_FITS).to_csv(RUN/'tables/tree_ablation_selections.csv',index=False)
    b_eval[['rid','site','record','hour']+[f'{n}_p{h}' for n in ABLATION_COLS for h in (6,12)]].to_parquet(
        RUN/'scores/source_B_tree_ablation_predictions.parquet',index=False)
else:
    json_dump({'tree_ablations':'NOT RUN: RUN_TREE_ABLATIONS=False'},
              RUN/'supplement/tree_ablation_status.json')
print('Tree ablations trained:',list(ABLATION_COLS) if cfg.RUN_TREE_ABLATIONS else 'SKIPPED')


## Cell 32 — Optional, independently retrained temporal TCN ablations

The protocol specifies the following temporal sensitivity questions: Do learned **time order** and a 3× dilated kernel help beyond a pointwise kernel-1 model? Is performance stable when the maximum causal history is reduced to **24 hours or 6 hours**? How much value is added by **observation bits and recencies**, holding value channels and valid-hour padding masks? Each variant here trains a fresh TCN from A fitting records with the same seed, AdamW settings and A-only median/IQR scaler. The chosen epoch count is fixed by the primary OOF epoch rule rather than tuned on B.

Because four additional full-dataset neural fits can exceed a Kaggle GPU session, `RUN_TCN_ABLATIONS=False` by default. Set it **True in Cell 2 before Run All** for a separate full protocol execution; their results are **explicitly absent** when skipped. They are TCN-alone comparisons, not replacement stacks (a replacement stack would also require new variant-specific OOF base predictions and refitted meta heads). This distinction should appear in the journal supplement.

In [ ]:
TCN_ABLATION_SPECS={
    'kernel1_pointwise':{'kernel':1,'lookback':48,'no_obs':False},
    'history24':{'kernel':3,'lookback':24,'no_obs':False},
    'history6':{'kernel':3,'lookback':6,'no_obs':False},
    'no_explicit_observation_tcn':{'kernel':3,'lookback':48,'no_obs':True},
}
TCN_ABLATION_FITS=[]
if cfg.RUN_TCN_ABLATIONS:
    for name,options in tqdm(TCN_ABLATION_SPECS.items(),desc='Independent TCN ablation fits'):
        details=tcn_train(TRAIN_A,FINAL_SCALER,cfg.SEED+90,name,fixed_epochs=TCN_FINAL_EPOCH,
                          kernel=options['kernel'],lookback=options['lookback'],no_obs=options['no_obs'])
        pred=tcn_predict(details['checkpoint'],SITE_B,FINAL_SCALER,name,
                          lookback=options['lookback'],no_obs=options['no_obs'])
        cumulative=compose_intervals(pred)
        b_eval[f'{name}_p6']=cumulative[:,0];b_eval[f'{name}_p12']=cumulative[:,1]
        TCN_ABLATION_FITS.append({'name':name,**options,'trained_epochs':TCN_FINAL_EPOCH,
                                  'checkpoint':details['checkpoint']})
    ABLATION_RESULTS.extend(metrics_table(b_eval,list(TCN_ABLATION_SPECS)).to_dict(orient='records'))
    pd.DataFrame(TCN_ABLATION_FITS).to_csv(RUN/'tables/tcn_ablation_fits.csv',index=False)
    b_eval[['rid','site','record','hour']+[f'{n}_p{h}' for n in TCN_ABLATION_SPECS for h in (6,12)]].to_parquet(
        RUN/'scores/source_B_tcn_ablation_predictions.parquet',index=False)
else:
    json_dump({'tcn_ablations':'NOT RUN: RUN_TCN_ABLATIONS=False',
      'available_variants':list(TCN_ABLATION_SPECS)},RUN/'supplement/tcn_ablation_status.json')


## Cell 33 — Sampling-time diagnostic: terminal controls versus exactly ICU-hour-matched controls

The revised paper asks whether a misleading **last-hour negative control** can be distinguished by clock time rather than physiology. This **new, illustrative diagnostic** (not a reconstruction of historical notebook weights or the manuscript's 9,279/9,196 rows) identifies an incident case at its **last eligible hour at least 48h into its observed record**, and compares A-only `never-positive` controls at their terminal at-risk time. Then, for the same cases, select up to three distinct never-positive control records at the **case's exact ICU hour**, provided those controls have adequate observed history and future-negative follow-up. Split by record before any fitted classifier; no matched control is used in the main held-site analysis.

**Purpose of the chart:** show ICU-hour imbalance of terminal controls and reduction after exact-hour matching. If case horizons/record follow-up make matching sparse, report unmatched cases, unique controls, control reuse and selected case counts. The figure describes a specific timing bias and does **not** prove that every change in AUROC is leakage. AP from these artificially matched groups is not generalizable to the natural ICU prevalence.

In [ ]:
# Pure sampling diagnostic; never feeds training or changes main source-held-out keys.
A_meta=meta[meta.site=='A'].copy()
A_rec=rec[(rec.site=='A')&(rec.at_risk_rows>0)]
case_rids=A_rec.loc[A_rec.event==1,'rid'].to_numpy()
never_rids=set(A_rec.loc[A_rec.event==0,'rid'])
# Cases: last eligible at-risk row with >=48 observed prior hours, and known 6h target positive.
cases=A_meta[A_meta.rid.isin(case_rids)&(A_meta.y6==1)].copy()
cases['observed_history']=cases['hour_index']-cases['rid'].map(rec.set_index('rid')['origin'])+1
cases=cases[cases.observed_history>=48].sort_values(['rid','hour']).groupby('rid').tail(1)
terminal=A_meta[A_meta.rid.isin(never_rids)].sort_values(['rid','hour']).groupby('rid').tail(1)
terminal=terminal[(terminal.hour_index-terminal.rid.map(rec.set_index('rid')['origin'])+1)>=48]
by_hour=defaultdict(list)
origin_by_rid=rec.set_index('rid')['origin'].to_dict()
for r in A_meta[A_meta.rid.isin(never_rids)&(A_meta.y6==0)].itertuples(index=False):
    if r.hour_index-origin_by_rid[r.rid]+1>=48:
        by_hour[int(r.hour)].append((int(r.rid),int(r.hour)))
rng=np.random.default_rng(cfg.SEED)
selected_case_hours=[];matched_control_hours=[];matched_control_ids=[];unmatched=0
for row in tqdm(cases.itertuples(index=False),total=len(cases),desc='A-only exact ICU-hour matching'):
    pool=list({rid for rid,t in by_hour.get(int(row.hour),[]) if rid != row.rid})
    if not pool:
        unmatched+=1;continue
    picked=rng.choice(pool,size=min(3,len(pool)),replace=False).tolist()
    selected_case_hours.append(row.hour)
    matched_control_hours.extend([row.hour]*len(picked))
    matched_control_ids.extend(picked)
sampling_qc={'candidate_legacy_like_cases':int(len(cases)),
             'case_hours_with_any_match':int(len(selected_case_hours)),
             'unmatched_cases':int(unmatched),'available_terminal_controls':int(len(terminal)),
             'matched_control_selections':int(len(matched_control_ids)),
             'unique_matched_controls':int(len(set(matched_control_ids))),
             'control_reuses':int(len(matched_control_ids)-len(set(matched_control_ids))),
             'diagnostic_is_not_historical_replication':True}
json_dump(sampling_qc,RUN/'tables/A_only_sampling_diagnostic.json')
plt.figure(figsize=(9,5))
if len(cases) and len(terminal):
    high=np.percentile(np.r_[cases.hour.to_numpy(),terminal.hour.to_numpy()],99)
    bins=np.arange(0,max(5,int(high)+3),3)
    plt.hist(cases.hour,bins=bins,density=True,histtype='step',label='Cases, legacy-like times',linewidth=2)
    plt.hist(terminal.hour,bins=bins,density=True,histtype='step',label='Never-positive terminal controls',linewidth=2)
    if matched_control_hours:
        plt.hist(matched_control_hours,bins=bins,density=True,histtype='step',
                 label='Same-ICU-hour matched controls',linewidth=2)
    plt.legend(fontsize=8)
else:
    plt.text(.5,.5,'No 48-hour-history incident cases available for this diagnostic',
             transform=plt.gca().transAxes,ha='center',va='center')
plt.xlabel('Observed ICU hour (for diagnostic matching only)')
plt.ylabel('Density')
plt.title('A-only diagnostic of terminal-time versus exact-hour control selection')
save_fig('08_sampling_terminal_vs_matched_hour')


## Cell 34 — Optional **two-XGBoost stacking** versus heterogeneous XGBoost–TCN

The architecture protocol explicitly identifies a **state-only XGBoost + trajectory-only XGBoost** stack as an alternative representation control. The cell below offers a compute-expensive, **fully OOF** second system. To keep it feasible on Kaggle, its sensitivity protocol uses prespecified **depth 4 and 200 rounds**, rather than the full depth grid. Each held fold is scored by state/trajectory boosters trained on the other four record groups; an L2 logistic meta learner sees only these OOF logits. Final state/trend boosters then train on A fitting, A policy chooses the same kind of false-alarm budget threshold, and B is scored once.

**Critical scientific labeling:** This secondary architecture intentionally has a **different hyperparameter search budget** than the primary tuned full-feature tree and should not be presented as a perfectly compute-matched contest. Turn on `RUN_TWO_TREE_STACK=True` only before a fresh Run All; when off, its figures and p-values are **not claimed to exist**. The main stack always remains XGBoost + causal TCN, per the protocol.

In [ ]:
# Set CFG.RUN_TWO_TREE_STACK=True in Cell 2 before a full experimental rerun.
# This additional ablation is intentionally off in the default session.
RUN_TWO_TREE_STACK=cfg.RUN_TWO_TREE_STACK
TWO_TREE_STATUS={'run':False,'reason':'Optional compute-heavy OOF architecture control was not requested in default run'}
if RUN_TWO_TREE_STACK:
    subset_names=['state_only','trajectory_only']
    fixed_depth,fixed_rounds=4,200
    oof_tree_views={s:np.full((len(meta),2),np.nan,dtype=np.float32) for s in subset_names}
    models_tree={}
    for fold in tqdm(range(cfg.N_FOLDS),desc='Two-tree OOF architecture comparison'):
        held=np.flatnonzero((meta.split.to_numpy()=='fit')&(FOLD==fold))
        train=np.flatnonzero((meta.split.to_numpy()=='fit')&(FOLD!=fold))
        for name in subset_names:
            for j in (0,1):
                bo,_,_=train_subset_booster(ABLATION_COLS[name],train,None,j,fixed_depth,
                        fixed_rounds,cfg.SEED+fold,f'two-tree fold{fold} {name} I{j+1}')
                bo.save_model(RUN/'fit'/f'two_tree_oof_fold{fold}_{name}_i{j+1}.json')
                oof_tree_views[name][held,j]=subset_predict(bo,held,ABLATION_COLS[name])
                del bo
    tree_metas=[]
    for j in (0,1):
        eligible=TRAIN_A[np.isfinite(ALL_LABELS[TRAIN_A,j])]
        z=np.column_stack([clipped_logit(oof_tree_views[name][eligible,j]) for name in subset_names])
        meta_tree=LogisticRegression(C=1,solver='lbfgs',max_iter=1000)
        meta_tree.fit(z,ALL_LABELS[eligible,j].astype(int))
        tree_metas.append(meta_tree)
    preds={k:[] for k in ('A','B')}
    for name in subset_names:
        out=[]
        for j in (0,1):
            bo,_,_=train_subset_booster(ABLATION_COLS[name],TRAIN_A,None,j,
                      fixed_depth,fixed_rounds,cfg.SEED,f'two-tree final {name} I{j+1}')
            bo.save_model(RUN/'models'/f'two_tree_{name}_i{j+1}.json')
            out.append((subset_predict(bo,POLICY_A,ABLATION_COLS[name]),
                        subset_predict(bo,SITE_B,ABLATION_COLS[name])))
            del bo
        models_tree[name]=out
    for tag,destination in [('A',policy_eval),('B',b_eval)]:
        intervals=[]
        select=0 if tag=='A' else 1
        for j in (0,1):
            z=np.column_stack([clipped_logit(models_tree[name][j][select]) for name in subset_names])
            intervals.append(tree_metas[j].predict_proba(z)[:,1])
        cumulative=compose_intervals(np.column_stack(intervals))
        destination['two_tree_stack_p6']=cumulative[:,0]
        destination['two_tree_stack_p12']=cumulative[:,1]
    for h in (6,12):
        p=prepare_replay(policy_eval,h);column=f'two_tree_stack_p{h}'
        ts=np.unique(np.r_[np.quantile(p['q'][column],np.linspace(0,1,101)),1.0000001])
        curve=[replay_prepared(p,column,float(t),h) for t in ts]
        good=[x for x in curve if x['false_episodes_per_100_evaluable_h']<=cfg.ALERT_FALSE_BUDGET]
        chosen=sorted(good,key=lambda x:(-x['event_sensitivity'],x['emitted_episodes'],-x['threshold']))[0]
        POLICY_THRESHOLDS[f'two_tree_stack_{h}']=chosen['threshold']
        v=replay_prepared(prepare_replay(b_eval,h),column,chosen['threshold'],h)
        v.update(model='two_tree_stack',horizon=h,site='B')
        B_ALERT_ROWS.append(v)
    pd.DataFrame(B_ALERT_ROWS).to_csv(RUN/'tables/held_site_B_frozen_alerts.csv',index=False)
    json_dump({k:v for k,v in POLICY_THRESHOLDS.items() if k.startswith('two_tree_stack_')},
              RUN/'policy/two_tree_thresholds.json')
    extra=metrics_table(b_eval,['two_tree_stack'])
    extra.to_csv(RUN/'tables/two_tree_vs_heterogeneous_metrics.csv',index=False)
    b_eval[['rid','site','record','hour','two_tree_stack_p6','two_tree_stack_p12']].to_parquet(
        RUN/'scores/source_B_two_tree_stack_predictions.parquet',index=False)
    json_dump([{'interval':j+1,'intercept':float(m.intercept_[0]),
                 'coefficients':m.coef_[0].tolist()} for j,m in enumerate(tree_metas)],
              RUN/'models/two_tree_meta_coefficients.json')
    TWO_TREE_STATUS={'run':True,'fixed_depth':fixed_depth,'fixed_rounds':fixed_rounds,
                     'n_models_per_OOF_fold':4,'two_tree_meta_from_OOF':True}
json_dump(TWO_TREE_STATUS,RUN/'supplement/two_tree_architecture_status.json')


## Cell 35 — Global model explainability: XGBoost gain importances and subgroup support

**Tree feature importance:** compute mean normalized *gain importance* over the two frozen final XGBoost interval boosters. Report the top feature names and summarize contributions by physiologic variable/family (state vs trajectory). Gain is a model-dependent measure of reliance, **not** proof of clinical causation. The most correct next step for mechanistic interpretation would be conditional/permutation-group importance within matched time strata, which is not a feature of the core predictor.

**Subgroup audit:** tabulate AP and sample/event counts by early/late ICU observation hour (`<48` versus `≥48`), age (`<65` versus `≥65`, plus missing), coded sex values (`0`,`1`,`missing`), and raw measurement-history sparsity quartiles defined using A fitting records only. Do not report an AUROC for a subgroup with a single class. Small event counts should be shown as descriptive with uncertainty rather than fairness or clinical equivalence assertions.

In [ ]:
imp=defaultdict(float)
for bo in FINAL_XGB:
    gains=bo.get_score(importance_type='gain')
    total=sum(gains.values())
    for k,v in gains.items():imp[k]+=float(v)/max(total,1e-9)/2
imp_df=pd.DataFrame([{'feature':k,'mean_normalized_gain':v,
                      'view':'state' if k in STATE_COLS else 'trajectory',
                      'clinical_variable':k.split('__')[0]} for k,v in imp.items()], columns=["feature","mean_normalized_gain","view","clinical_variable"]).sort_values(
                          'mean_normalized_gain',ascending=False)
imp_df.to_csv(RUN/'tables/xgb_feature_gain_importance.csv',index=False)
if not imp_df.empty:
    top=imp_df.head(24).iloc[::-1]
    plt.figure(figsize=(9,8))
    plt.barh(top.feature,top.mean_normalized_gain)
    plt.xlabel('Average normalized gain from final I1 and I2 boosters')
    plt.title('Top XGBoost engineered feature importances (predictive only)')
    plt.yticks(fontsize=7);save_fig('09_xgb_feature_importance')

B_pred=b_eval.copy()
B_pred['position_in_record']=B_pred.hour_index if 'hour_index' in B_pred.columns else B_pred.hour
B_pred['early_vs_late']=np.where(B_pred.hour<48,'hour<48','hour>=48')
# Stage-available Age/Gender are model inputs only; constructing subgroups is descriptive.
B_features=X[SITE_B,102:104]
B_pred['age_group']=np.where(np.isnan(B_features[:,0]),'missing',
                             np.where(B_features[:,0]<65,'under_65','65_plus'))
B_pred['dataset_coded_sex']=np.where(np.isnan(B_features[:,1]),'missing',
                                     B_features[:,1].astype(str))
measurement_count=np.sum(X[:,[FEATURE_COLS.index(f'{v}__measured_now') for v in DYN]],axis=1)
fit_edges=np.quantile(measurement_count[TRAIN_A],[.25,.5,.75])
B_pred['observation_density']=np.searchsorted(fit_edges,measurement_count[SITE_B],side='right').astype(str)
SUBGROUP=[]
for group in ('early_vs_late','age_group','dataset_coded_sex','observation_density'):
    for category,part in B_pred.groupby(group):
        for h in (6,12):
            known=part[f'y{h}'].notna(); yt=part.loc[known,f'y{h}'].to_numpy(dtype=int)
            if len(yt)==0:continue
            p=part.loc[known,f'stack_p{h}'].to_numpy()
            SUBGROUP.append({'group':group,'category':str(category),'horizon':h,
                             'n_evaluable':len(yt),'n_positive':int(yt.sum()),
                             'n_records':part.loc[known,'rid'].nunique(),
                             'AP':float(average_precision_score(yt,p)) if yt.sum()>0 else np.nan,
                             'AUROC':float(roc_auc_score(yt,p)) if len(np.unique(yt))==2 else np.nan})
pd.DataFrame(SUBGROUP).to_csv(RUN/'tables/held_site_B_subgroups.csv',index=False)


## Cell 36 — Paired ablation forest plot and clustered uncertainty for secondary contrasts

Forest plots visualize **differences**, not a misleading ranking of AP values from different evaluation denominators. Compare the main `stack` to `calibrated_xgb` with the **full 2,000-record bootstrap primary interval**; show secondary paired 6-hour AP contrasts against calibrated TCN, the fixed equal-average, and raw XGBoost using separate record bootstrap draws. All use identical B keys, so the differences are paired. Optional tree/TCN representation ablations are reported in a separate table and marked exploratory.

**Compute setting:** the main 2,000 draws from Cell 27 are authoritative for primary inference; use 150 (pilot) or 300 (full) for visual, explicitly exploratory secondary uncertainty to keep Kaggle runtime bounded. Do not call a secondary difference significant solely because it has a favorable point estimate. If a CI spans zero, report inconclusive evidence. Keep exploratory comparisons out of a single uncorrected confirmatory claim.

In [ ]:
FOREST=[{'contrast':'Stack − calibrated full XGB (PRIMARY)',**PRIMARY_CI}]
secondary=['calibrated_tcn','equal_average','raw_xgb']
if RUN_TWO_TREE_STACK:secondary.append('two_tree_stack')
sec_n=150 if cfg.RUN_MODE=='pilot' else 300
for comp in tqdm(secondary,desc='Secondary paired AP CIs'):
    ci,_=paired_ap_bootstrap(b_eval,'stack',comp,6,sec_n,cfg.SEED+103+secondary.index(comp),
                  f'Secondary stack versus {comp}')
    FOREST.append({'contrast':f'Stack − {comp} (exploratory)',**ci})
forest=pd.DataFrame(FOREST)
forest.to_csv(RUN/'tables/paired_architecture_ablation_forest.csv',index=False)
fig,ax=plt.subplots(figsize=(9,max(4,len(forest)*.7+1)))
ypos=np.arange(len(forest))
ax.hlines(ypos,forest['CI_low'],forest['CI_high'],color='#468392',linewidth=2)
ax.scatter(forest['AP_difference'],ypos,color='#193d54',zorder=3)
ax.axvline(0,color='black',linestyle='--')
ax.set_yticks(ypos,forest.contrast)
ax.invert_yaxis();ax.set_xlabel('Paired six-hour B average-precision difference (95% record-bootstrap CI)')
ax.set_title('Incremental architecture value: primary and exploratory comparisons')
save_fig('10_ablation_paired_AP_forest')
if ABLATION_RESULTS:
    pd.DataFrame(ABLATION_RESULTS).to_csv(RUN/'tables/retrained_representation_ablation_metrics.csv',index=False)


## Cell 37 — Diagnostic figures: learning curves, lead time and events versus eligible hours

**Training behavior:** plot TCN training and internal grouped-validation masked log loss, and display the XGBoost depth/round selection table. Loss curves are particularly important for checking whether best-epoch selection happened before convergence or severe overfitting. Different folds have different training patient groups and thus may have different optimal stopping epochs. Do not interpret better training loss as held-site improvement.

**Detection:** chart source-B incident-event count and frozen-policy detected-event count for both horizons, and represent lead times **only among detected events**; an impressive median lead time among 10 detected events must not be described as applying to all incident cases. All primary alert time units are measured relative to the **first positive benchmark label**, not to independently timestamped clinical onset.

In [ ]:
history=pd.DataFrame(TCN_HISTORY)
plt.figure(figsize=(9,5.5))
for tag,g in history.groupby('tag'):
    if 'selection' in tag:
        plt.plot(g.epoch,g.valid_masked_logloss,label=tag,linewidth=1)
plt.xlabel('TCN training epoch');plt.ylabel('Validation masked interval log loss')
plt.title('Inner record-grouped early stopping by OOF fold')
plt.legend(fontsize=7);save_fig('11_tcn_fold_early_stopping')

ab=ALERT_RESULTS[ALERT_RESULTS.model.isin(['calibrated_xgb','stack','raw_tcn'])]
fig,ax=plt.subplots(figsize=(9,5))
bar=ab.pivot(index='horizon',columns='model',values='detected_event_records')
bar.plot.bar(ax=ax)
ax.set_title('Detected incident records at A-frozen thresholds, held-out B')
ax.set_xlabel('Forecast window (hours)');ax.set_ylabel('Distinct events detected')
save_fig('12_detected_events')
fig,ax=plt.subplots(figsize=(8,5))
for model in ('calibrated_xgb','stack','raw_tcn'):
    sub=ALERT_RESULTS[ALERT_RESULTS.model==model].sort_values('horizon')
    if not sub.empty:
        ax.plot(sub.horizon,sub.lead_time_median_h,marker='o',label=model)
ax.set_xticks([6,12]);ax.set_xlabel('Forecast window (hours)')
ax.set_ylabel('Median first qualifying alert → label-transition (hours)')
ax.set_title('Lead time among DETECTED events only')
ax.legend();save_fig('13_alert_lead_time_detected_only')
# Real first-detection lead-time distributions (not medians alone).
for h in (6,12):
    p=prepare_replay(b_eval,h)
    plt.figure(figsize=(8,5))
    any_detected=False
    for model in ('calibrated_xgb','stack','raw_tcn'):
        z=replay_numba(p['hours'],p['starts'],p['q'][f'{model}_p{h}'].to_numpy(dtype=np.float32),
                       p['events'],p['last'],float(POLICY_THRESHOLDS[f'{model}_{h}']),
                       int(h),int(cfg.ALERT_COOLDOWN_HOURS),p['known'])
        leads=z[-1];leads=leads[np.isfinite(leads)]
        if len(leads):
            any_detected=True
            plt.hist(leads,bins=np.arange(.5,h+1.6,1),alpha=.45,label=f'{model} (n={len(leads)})')
    if any_detected: plt.legend(fontsize=8)
    else: plt.text(.5,.5,'No qualifying detected events at frozen thresholds',
                   transform=plt.gca().transAxes,ha='center',va='center')
    plt.xlabel('Earliest qualifying alert to FIRST POSITIVE LABEL (hours)')
    plt.ylabel('Number of detected incident records')
    plt.title(f'Lead time distribution among detected B cases, {h}h policy')
    save_fig(f'14_first_alert_lead_time_histogram_{h}h')


## Cell 38 — Architecture diagram for the journal and complete reproducibility manifest

The notebook's prose diagram documents **what each input means**, while this cell produces the publication-ready graphical overview. Clearly distinguish (i) ingestion of raw A/B PSV records; (ii) causal 512-feature tree view and 48×102 temporal view; (iii) record-disjoint fivefold unseen OOF fitting; (iv) two per-interval logistic heads and survival-consistent six/twelve-hour composition; (v) separate A-only threshold tuning; and (vi) final B-only frozen external evaluation.

**Audit package:** record all counts, chosen model depths and rounds, TCN best epochs, final scaler hash, record split hash, feature-schema hash, input inventory hash, run mode, source B's positive prevalence and whether each optional experiment ran. Provide one methods/limitations file for the article supplement. Figures should have literal captions that say **benchmark label-transition**, not “sepsis onset at 12 hours,” unless independently adjudicated onset labels exist.

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
fig,ax=plt.subplots(figsize=(15,9))
ax.set_xlim(0,15);ax.set_ylim(0,10);ax.axis('off')
def diagram_box(x,y,w,h,text,color='#eaf3fc'):
    p=FancyBboxPatch((x,y),w,h,boxstyle='round,pad=0.08,rounding_size=0.15',
                     ec='#30465c',fc=color,lw=1.3)
    ax.add_patch(p)
    ax.text(x+w/2,y+h/2,text,ha='center',va='center',fontsize=10,wrap=True)
def arrow(a,b):
    ax.add_patch(FancyArrowPatch(a,b,arrowstyle='-|>',mutation_scale=14,lw=1.5,color='#4c5966'))
diagram_box(5,8.75,5,0.75,'PhysioNet 2019 A/B raw PSV  →  manifest, QC, uninterrupted labels')
diagram_box(1,6.95,5.6,1,'Tabular 512 = 104 state + 408 6/24h summaries\nXGBoost, two interval boosters')
diagram_box(8.45,6.95,5.6,1,'48 × 102 causal sequence + 4 static features\nTCN 102→64, dilations 1/2/4/8, 68→32→2')
diagram_box(5,5.2,5,0.85,'5 grouped OOF folds + inner stop sets (source A fitting only)')
diagram_box(4.7,3.55,5.6,0.95,'Two L2 logistic heads on OOF base logits\n(a, b) = interval-1, conditional interval-2 risks','#ecf9ed')
diagram_box(4.7,2.18,5.6,0.85,'P6 = a; P12 = a + (1−a)b','#ecf9ed')
diagram_box(0.5,0.6,6.5,0.95,'15% source-A policy subset\nFrozen threshold: ≤2 false episodes / 100 evaluable h')
diagram_box(8,0.6,6.5,0.95,'Source B only after final scoring\nPaired 6h AP, 12h, calibration, alert replay, CIs')
for aa,bb in [((7.5,8.75),(3.8,7.98)),((7.5,8.75),(11.3,7.98)),
              ((3.8,6.95),(5.6,6.05)),((11.3,6.95),(9.4,6.05)),
              ((7.5,5.2),(7.5,4.5)),((7.5,3.55),(7.5,3.03)),
              ((6.1,2.18),(4.3,1.55)),((8.9,2.18),(10.5,1.55))]:arrow(aa,bb)
ax.set_title('Cross-site prediction of the first observed benchmark sepsis-label transition',fontsize=15,pad=15)
save_fig('00_journal_architecture_overview')

SCHEMA_HASH=hashlib.sha256(json.dumps(FEATURE_COLS,separators=(',',':')).encode()).hexdigest()
RUN_REGISTRY={
 'study':'Retrospective forecasting of first observed sepsis-label transition',
 'source':'PhysioNet/CinC Challenge 2019 v1.0.0 via the requested Kaggle mirror',
 'study_claims':'Benchmark label-transition only; not independently validated onset',
 'run_utc':datetime.now(timezone.utc).isoformat(),
 'run_mode':cfg.RUN_MODE,'pilot_results_not_for_publication':cfg.RUN_MODE!='full',
 'config':asdict(cfg),'inventory_sha256':inv_hash,'split_sha256':split_hash,
 'feature_schema_sha256':SCHEMA_HASH,'feature_n':512,
 'xgb_feature_views':{'state':104,'trajectory':408},
 'tcn':{'shape':[48,102],'static_features':4,'width':cfg.TCN_WIDTH,
        'residual_dilations':[1,2,4,8], 'kernel':3,'params':count_params(CausalTCN()),
        'final_selected_epoch':TCN_FINAL_EPOCH},
 'final_xgb_models':FINAL_XGB_CONFIG,
 'primary_endpoint':'B six-hour paired AP difference: stack - calibrated full XGB',
 'primary_result':PRIMARY_CI,
 'A_only_alarm_budget':cfg.ALERT_FALSE_BUDGET,
 'tree_ablation_run':cfg.RUN_TREE_ABLATIONS,
 'tcn_ablation_run':cfg.RUN_TCN_ABLATIONS,
 'two_tree_stack_run':RUN_TWO_TREE_STACK,
 'reverse_site_run':False,'independent_clinical_onset_validation_run':False,
 'training_seed_repetition_run':False,
 'source_B_fitted_or_calibrated_on':False,
 'warnings':XGB_DEVICE_LOG,
 'A_B_cohort':cohort.to_dict(orient='records'),
 'known_landmarks':summary.to_dict(orient='records'),
}
json_dump(RUN_REGISTRY,RUN/'supplement/run_registry.json')


## Cell 39 — Reload checkpoints, inspect all model/figure outputs and render manuscript tables

**Release gate:** a saved estimator is not an inference artifact until it can be loaded and reproduce a fixed held-out score. Reload the two XGBoost JSON boosters and the PyTorch checkpoint, then check a small *A policy* batch against the originally generated scores within numerical tolerance. Confirm that every source-B output has the right key count, no duplicates and cumulative risk ordering; that OOF scores exist exactly once per fitting record-hour; that every primary figure/table file exists; and that the 6-hour AP contrast in `run_registry.json` equals the archived metrics table to within floating-point precision.

**Human-readable report:** generate journal-ready plain-text and CSV tables from the exact saved predictions and alarms, including denominators and whether optional ablation results exist. The notebook **does not prefill invented numerical improvements**. If any gate fails, stop instead of packaging an apparently publication-ready ZIP.

In [ ]:
for j in (0,1):
    loaded=xgb.Booster()
    loaded.load_model(RUN/'models'/f'xgboost_interval_{j+1}.json')
    i=POLICY_A[:min(8,len(POLICY_A))]
    expected=predict_xgb(FINAL_XGB[j],i)
    restored=predict_xgb(loaded,i)
    np.testing.assert_allclose(expected,restored,rtol=1e-5,atol=1e-5)
    del loaded
sample_i=POLICY_A[:min(4,len(POLICY_A))]
# The reference predictions were frozen before checkpoint verification.
reference = policy_scores.set_index(['site','record','hour'])
keys = pd.MultiIndex.from_frame(meta.loc[sample_i,['site','record','hour']])
stored = reference.loc[keys,['raw_tcn_p6','raw_tcn_p12']].to_numpy()
restored = tcn_predict(TCN_FINAL_PATH,sample_i,FINAL_SCALER,'Reload final TCN')
np.testing.assert_allclose(compose_intervals(restored),stored,rtol=1e-5,atol=1e-6)
assert len(oof)==len(TRAIN_A) and not oof.duplicated(['site','record','hour']).any()
assert len(b_scores)==len(SITE_B)
assert (RUN/'tables/held_site_B_main_metrics.csv').exists()
assert (RUN/'policy/thresholds.json').exists()
assert len(list((RUN/'figures').glob('*.png')))>=14
main6=RESULTS[(RESULTS.horizon_h==6)&(RESULTS.method.isin(['stack','calibrated_xgb']))]
check=float(main6.set_index('method').loc['stack','average_precision']-
            main6.set_index('method').loc['calibrated_xgb','average_precision'])
assert abs(check-PRIMARY_CI['AP_difference'])<1e-9
assert len(RUN_REGISTRY['final_xgb_models'])==2

report_lines=[
'# Reproducible cross-site sepsis label-transition study',
'',f'Run mode: **{cfg.RUN_MODE}**',
'Clinical interpretation: predictions target the first observed positive benchmark label, not independently timestamped sepsis onset.',
'Primary external source: B, with entire records held out during model fitting and policy selection.',
'',f'Primary paired 6h AP difference (stack − calibrated XGB): {check:.6f}',
 f"Record-bootstrap {95}% interval: [{PRIMARY_CI['CI_low']:.6f}, {PRIMARY_CI['CI_high']:.6f}]",
 f"Defined bootstrap samples: {PRIMARY_CI['n_defined']}/{PRIMARY_CI['n_boot']}",
 '', '## Main held-site B methods (natural-prevalence landmark estimates)',
 RESULTS.to_string(index=False,float_format=lambda x:f'{x:.5f}'),
 '', '## Frozen alert policy operating points on B',
 ALERT_RESULTS.to_string(index=False,float_format=lambda x:f'{x:.5f}'),
 '', '## Study limitations',
 '- Labels are benchmark-derived and known to precede the Challenge event; interpret only as label-transition prediction.',
 '- Mirror A/B source sets are not a guarantee of independent unseen hospitals worldwide.',
 '- Public PSV records lack stable cross-record real-person identity links, true discharge/death and EHR result-availability timestamps.',
 '- Censored follow-up restricts the evaluated set; alarm episodes with unknown ascertainment are marked indeterminate.',
 '- Pilot mode numbers (if run) are strictly for software smoke testing, not the journal.',
 f'- Extended TCN ablations run: {cfg.RUN_TCN_ABLATIONS}. Two-tree OOF stack run: {RUN_TWO_TREE_STACK}.',
 '- Three independent training seeds, reverse-site replication, historic notebook-weight recovery and independent clinical-onset validation are not executed in the default notebook.',
 '', '## Required manuscript ethics and reporting',
 'Use TRIPOD+AI and PROBAST+AI checklists, report all denominators and source mappings, keep historical results in a separately labeled table, and distinguish primary from exploratory comparisons.',
]
(RUN/'supplement/JOURNAL_RESULTS_AND_LIMITATIONS.md').write_text('\n'.join(report_lines),encoding='utf-8')
print('Release checks PASS. PNG figures:',len(list((RUN/'figures').glob('*.png'))),
      'final booster JSON files:',len(list((RUN/'models').glob('xgboost*.json'))))


## Cell 40 — **FINAL OUTPUT CELL**: package **all weights, graphs, metrics and checksums into one ZIP**

**What is included:** XGBoost fold and final boosters (`.json`), TCN fold-selection/refit/final checkpoints (`.pt`), scaler JSON, stacking/calibration coefficients, feature dictionary and schema, raw file inventory with hashes, cohort/split audits, all prediction scores (including site B and OOF), all primary and run-enabled supplementary ablation tables, thresholds, environment, training logs, and every high-resolution PNG/SVG/PDF graph. A reproducibility manifest lists checksums for every archived file.

**What is intentionally excluded:** the original public PSV dataset and large intermediary `.npy` hourly and `.f32` tabular caches (regenerable from the linked Kaggle dataset), because including them would greatly inflate the download and expose raw clinical rows unnecessarily. The ZIP contains model **weights and result outputs**, not just a visual demonstration. Only the single `sepsis_research_results.zip` is written by this project to `/kaggle/working`. After verifying the archive, the temporary project cache is deleted, freeing Kaggle disk space. The notebook does not delete other people's files or other working folders.

**Publication check:** if an optional experiment was disabled, its status JSON says “NOT RUN”; there is no fabricated graph, weight, p-value, or test result. Do not confuse the local notebook file (distributed separately) with the Kaggle results ZIP generated after actually running on GPU.

In [ ]:
import zipfile
# The project-specific temporary folder is the only directory touched by cleanup.
include_paths=[]
for p in sorted(RUN.rglob('*')):
    if not p.is_file() or p.name=='archive.partial': continue
    relative=p.relative_to(RUN)
    if relative.suffix in ('.npy','.f32') or p.name=='ARCHIVE_CHECKSUMS.json': continue  # large raw/feature caches, not trained weights
    if p.name=='raw_manifest.parquet' or relative.parts[0] in ('models','fit','scores','policy',
                                                 'figures','tables','supplement','audit','data'):
        include_paths.append(p)
    elif relative.parts[0] not in ('data',): include_paths.append(p)
files_sha={str(p.relative_to(RUN)):sha256_file(p) for p in tqdm(include_paths,desc='Hash final research artifacts')}
json_dump({'files_sha256':files_sha,'total_files':len(include_paths),
           'excluded_intermediate_arrays':True,'full_clinical_psv_inputs_included':False},
          RUN/'supplement/ARCHIVE_CHECKSUMS.json')
include_paths.append(RUN/'supplement/ARCHIVE_CHECKSUMS.json')
with tempfile.NamedTemporaryFile(dir=BASE, prefix='.sepsis_', suffix='.partial', delete=False) as pending:
    pending_zip=Path(pending.name)
try:
    with zipfile.ZipFile(pending_zip,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6,
                         allowZip64=True) as out:
        for p in tqdm(include_paths,desc='Compress final ONLY ZIP output'):
            out.write(p,arcname=str(p.relative_to(RUN)))
    with zipfile.ZipFile(pending_zip,'r') as z:
        damaged=z.testzip()
        if damaged is not None: raise RuntimeError(f'Damaged ZIP member: {damaged}')
        archive_names=set(z.namelist())
        assert 'models/tcn_final.pt' in archive_names
        assert 'models/xgboost_interval_1.json' in archive_names
        assert 'models/xgboost_interval_2.json' in archive_names
        assert 'figures/00_journal_architecture_overview.pdf' in archive_names
        assert 'supplement/ARCHIVE_CHECKSUMS.json' in archive_names
    os.replace(pending_zip, OUTPUT_ZIP)
finally:
    pending_zip.unlink(missing_ok=True)
size_mb=OUTPUT_ZIP.stat().st_size/2**20
print(f'VALIDATED ZIP: {OUTPUT_ZIP} | {size_mb:,.1f} MiB | {len(include_paths)} files')
# Post-packaging cleanup of this experiment's dedicated temporary folder only.
shutil.rmtree(RUN)
print('Dedicated temporary research cache removed. Final project output:',OUTPUT_ZIP)
display(FileLink(os.path.relpath(OUTPUT_ZIP, Path.cwd())))


---
## Reading the result ZIP after training

`models/` contains the **frozen deployment predictor** (two tree JSON boosters, one TCN state-dict checkpoint, its scaler, meta/calibration coefficients and selected epoch/rounds). `fit/` contains **all OOF-fold weight checkpoints and training histories**. `scores/` contains unaltered A-policy/B scoring outputs and unseen OOF predictions. `tables/` includes the paired primary test, calibration, alarm episodes, ablations, cohort counts and feature importances. `figures/` holds every figure as PNG, SVG and PDF. `policy/` stores A-chosen thresholds and burden curves; `supplement/` holds limitations and checksums; `data/` holds dataset provenance, source hashes and 512-column schema, without the huge raw feature arrays.

**Why original PSV data and `.npy` feature caches are excluded:** the original dataset remains accessible through the linked data handle, while saving multi-gigabyte raw arrays in the ZIP would complicate the transfer. The exact record and feature extraction are implemented in the notebook above and tied to manifest/split/schema hashes.

**Uncompleted research is not completed by the notebook's existence.** In particular, 3-seed stability, independent clinical-onset validation, reverse-site experiment, reproducing historical checkpoint weights and optional deep ablations require additional runs/inputs and should not appear as measured results if unrun. A publishable journal paper still needs external validation of the claims, careful methods reporting and a defensible discussion of negative results.

---
# Paper reconstruction and archived analysis

This expanded notebook keeps the original numbered research pipeline and adds an auditable review of the analyses reported in the revised paper. The original notebook trains the primary XGBoost and temporal convolutional network models. The accompanying results archive contains additional review tables and checkpoints.

**Evidence boundary.** This notebook does not claim to reconstruct every historical command or exact analysis implementation. In particular, the archive does not contain the held source B row level prediction stream required to rerun same hour concordance, crossed score policies or all paired alert replays. The cells below read and summarize the archived review tables for those analyses; they do not present the summaries as newly computed model results. The optional MB TCN benchmark and parts of the source exchange experiment also require additional original training inputs.

Place `sepsis_review_full_A_to_B.zip` beside this notebook, or set `ARCHIVE_PATH` in Cell 43. Run the original training cells only when you intend to retrain models. The archive review cells can be run independently and do not require downloading the clinical data.


## Cell 41 — Locate the supplied experiment archive

**Purpose.** Find the result ZIP from the project folder or a mounted Kaggle input. The archive is read without extracting it, so this cell does not overwrite or delete files.


In [ ]:
from pathlib import Path
import zipfile
import json
import io

ARCHIVE_CANDIDATES = [
    Path.cwd() / 'sepsis_review_full_A_to_B.zip',
    Path.cwd().parent / 'sepsis_review_full_A_to_B.zip',
    Path('/kaggle/input/sepsis-review-full-a-to-b/sepsis_review_full_A_to_B.zip'),
]
ARCHIVE_PATH = next((p for p in ARCHIVE_CANDIDATES if p.is_file()), None)
if ARCHIVE_PATH is None:
    raise FileNotFoundError(
        'Place sepsis_review_full_A_to_B.zip beside the notebook or set ARCHIVE_PATH.'
    )

archive = zipfile.ZipFile(ARCHIVE_PATH)
archive_names = set(archive.namelist())
required = {
    'supplement/run_registry.json',
    'tables/held_site_B_main_metrics.csv',
    'tables/review_time_metrics.csv',
    'tables/review_time_contrasts.csv',
    'tables/review_tcn_ablation.csv',
    'tables/review_seeds.csv',
    'tables/review_schedule.csv',
}
missing = required - archive_names
if missing:
    raise ValueError(f'Archive is missing required review files: {sorted(missing)}')
print(f'Archive: {ARCHIVE_PATH} ({len(archive_names):,} members)')


## Cell 42 — Read the run registry and record completed experiments

The registry records which training runs the archived package reports as completed. A saved table is evidence that results were archived; it does not, by itself, establish that this notebook generated every table.


In [ ]:
run_registry = json.loads(archive.read('supplement/run_registry.json'))
archive_config = json.loads(archive.read('config.json'))

experiment_status = {
    'run_mode': run_registry.get('run_mode'),
    'primary_endpoint': run_registry.get('primary_endpoint'),
    'source_B_fitted_or_calibrated_on': run_registry.get(
        'source_B_fitted_or_calibrated_on'
    ),
    'TCN_ablation_run': run_registry.get('tcn_ablation_run'),
    'reverse_site_run': run_registry.get('reverse_site_run'),
    'training_seed_repetition_run': run_registry.get(
        'training_seed_repetition_run'
    ),
    'independent_clinical_onset_validation_run': run_registry.get(
        'independent_clinical_onset_validation_run'
    ),
    'two_tree_stack_run': run_registry.get('two_tree_stack_run'),
}
print(json.dumps(experiment_status, indent=2))


## Cell 43 — Load archived performance and model review tables

The review tables are the recorded outputs used to prepare the manuscript. This cell loads them without changing their values.


In [ ]:
import pandas as pd
from IPython.display import display

def read_archive_csv(member):
    """Read one CSV member from the experiment ZIP."""
    with archive.open(member) as stream:
        return pd.read_csv(stream)

main_metrics = read_archive_csv('tables/held_site_B_main_metrics.csv')
time_metrics = read_archive_csv('tables/review_time_metrics.csv')
time_contrasts = read_archive_csv('tables/review_time_contrasts.csv')
tcn_ablation = read_archive_csv('tables/review_tcn_ablation.csv')
seed_metrics = read_archive_csv('tables/review_seeds.csv')
schedule_results = read_archive_csv('tables/review_schedule.csv')

print('Primary held source results')
display(main_metrics)
print('Elapsed time and same stage review metrics')
display(time_metrics)


## Cell 44 — Reconcile key paper values with archived metrics

This is a consistency check across the manuscript values and the archive. It is not a rerun of model inference or bootstrap sampling.


In [ ]:
primary_six_hour = main_metrics[main_metrics['horizon_h'].eq(6)].set_index('method')
stack_ap = float(primary_six_hour.loc['stack', 'average_precision'])
xgb_ap = float(primary_six_hour.loc['calibrated_xgb', 'average_precision'])
primary_ci = json.loads(archive.read('tables/primary_AP_bootstrap_CI.json'))

recorded = {
    'site_B_six_hour_stack_AP': stack_ap,
    'site_B_six_hour_calibrated_XGBoost_AP': xgb_ap,
    'paired_AP_difference': stack_ap - xgb_ap,
    'paired_bootstrap_CI_low': primary_ci['CI_low'],
    'paired_bootstrap_CI_high': primary_ci['CI_high'],
    'defined_bootstrap_replicates': primary_ci['n_defined'],
}
print(json.dumps(recorded, indent=2))
assert abs((stack_ap - xgb_ap) - primary_ci['AP_difference']) < 1e-10


## Cell 45 — Summarize the archived temporal network ablations

The archive records nine component contrasts and three training seeds. These tables are displayed as saved; this cell does not refit the variants.


In [ ]:
print('Component contrasts against the full temporal network')
display(tcn_ablation)

print('Repeated seed results')
display(seed_metrics)

print('Training schedule checks')
display(schedule_results)


## Cell 46 — Plot archived performance across elapsed ICU time

The chart uses archived point estimates. It shows pooled average precision by model and forecast horizon. For inference, use the paired bootstrap contrast table and retain its uncertainty intervals.


In [ ]:
import matplotlib.pyplot as plt

plot_data = time_metrics[
    time_metrics['hour_band'].eq('all')
    & time_metrics['method'].isin(
        ['stack', 'equal_average', 'calibrated_tcn', 'calibrated_xgb',
         'time_only', 'time_vitals', 'time_branches']
    )
].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for ax, horizon in zip(axes, (6, 12)):
    part = plot_data[plot_data['horizon_h'].eq(horizon)]
    part = part.sort_values('average_precision')
    ax.barh(part['method'], part['average_precision'], color='#148f86')
    ax.set(title=f'{horizon} hour forecast', xlabel='Average precision')
    ax.grid(axis='x', alpha=0.25)
fig.suptitle('Archived source B ranking results')
plt.show()


## Cell 47 — Reuse archived comparison tables and document reconstruction limits

The paper's same hour concordance, crossed alert distribution, reverse transfer and reference network benchmark are represented in the archived review materials, but not all source scores and training inputs needed to rerun them are present. Use the saved tables below as reported results. To reproduce those analyses from first principles, supply the original row level prediction files, exact split manifests, architecture adapter and training logs.


In [ ]:
optional_members = [
    name for name in sorted(archive_names)
    if name.startswith(('tables/review_', 'fit/repeat_seed_', 'fit/schedule_'))
]
print('Archived review artifacts:')
for member in optional_members:
    info = archive.getinfo(member)
    print(f'{member}  ({info.file_size:,} bytes)')

print('\nNot recorded as completed in the run registry:')
for key in ['reverse_site_run', 'independent_clinical_onset_validation_run',
            'two_tree_stack_run']:
    print(f'{key}: {run_registry.get(key)}')


## Cell 48 — Close the archive after review

Run this after the archive review cells if you want to release the ZIP handle.


In [ ]:
archive.close()
print('Archive review complete. The saved research outputs were read without modification.')
